# 基于顺序表的并行股票交易评估（答案）

顺序表是线性表的一种顺序存储实现。本实验以“股票交易评估”为背景，使用顺序表组织股票交易记录，在昇腾神经网络处理器（Neural Processing Unit，NPU）上实现并行求和算子，并通过成交量加权平均价格（Volume Weighted Average Price，VWAP）完成交易评估。

本实验学习大纲如下：

1. 环境准备：创建实验目录，检查数据文件并加载 CANN 环境；
2. 任务分析：分析股票交易评估问题、了解数据集、结构体和实验参数、并理解本任务中的并行实现；
3. 算法讲解：理解 Host/Device 侧分工、数据分块及两阶段并行求和的实现过程；
4. 代码构建与运行：完成数据结构定义、顺序表构建、CPU 参考算法、NPU 核函数、Host 调用及 CMake 构建；
5. 性能对比分析：使用默认交易编号分别运行 CPU 与 NPU 版本，展示交易评价结果并比较求和计算的平均耗时；
6. 实验总结：归纳顺序表的组织方式与并行求和的实现过程。

**注：** 本实验使用 Python Notebook 组织讲解和代码生成，主要算法使用 C++ 编写。NPU 部分使用昇腾计算架构（Compute Architecture for Neural Networks，CANN）提供的 Ascend C 开发工具。


---
## 1. 环境准备

本实验项目名称为 `stock_trading_evaluation_seq`。首先创建实验所需目录，并加载 Ascend CANN 环境变量。

目录划分如下：

- `trades.txt`：股票交易记录数据集文件；
- `Source/stock_trading_evaluation_seq/include`：股票记录、顺序表及公共接口；
- `Source/stock_trading_evaluation_seq/src`：数据读取、CPU 参考算法和测试程序；
- `Source/stock_trading_evaluation_seq/ascend`：Device 侧 Ascend C 核函数、Host 侧设备管理与调用程序；
- `Source/stock_trading_evaluation_seq/scripts`：构建与运行脚本；
- `Source/stock_trading_evaluation_seq/results`：正确性检查和计时结果。


In [ ]:
# 准备工程目录并检查同目录的数据文件，不执行 NPU 计算。
from pathlib import Path
import os
import platform
import shlex
import shutil
import subprocess

NOTEBOOK_DIR = Path.cwd().resolve()
DATA_FILE = NOTEBOOK_DIR / 'trades.txt'
WORK_DIR = NOTEBOOK_DIR / 'Source' / 'stock_trading_evaluation_seq'
for name in ['include', 'src', 'ascend', 'scripts', 'results']:
    (WORK_DIR / name).mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError('请将 trades.txt 放在本 Notebook 所在目录，并从该目录启动 Notebook。')

# CANN 根目录或架构子目录中应包含运行时头文件；ASC 编译条件由构建脚本检查。
def has_cann_runtime(path):
    return any((path / name).is_file() for name in
               ['include/acl/acl.h', 'runtime/include/acl/acl.h'])

candidates = []
for name in ['ASCEND_INSTALL_PATH', 'ASCEND_CANN_PACKAGE_PATH', 'ASCEND_TOOLKIT_HOME', 'ASCEND_HOME_PATH']:
    if os.environ.get(name):
        candidates.append(Path(os.environ[name]))
candidates += [Path('/usr/local/Ascend/cann'), Path('/usr/local/Ascend/cann/latest'),
               Path('/usr/local/Ascend/ascend-toolkit/latest')]
candidates += sorted(Path('/usr/local/Ascend/ascend-toolkit').glob('*/' + platform.machine() + '-linux'), reverse=True)
candidates += sorted(Path('/opt/conda/Ascend').glob('cann-*/' + platform.machine() + '-linux'), reverse=True)
CANN_PATH = next((p.resolve() for p in candidates if has_cann_runtime(p)), None)
if CANN_PATH is not None:
    env_file = next((p for p in [CANN_PATH / 'set_env.sh', CANN_PATH.parent / 'set_env.sh'] if p.is_file()), None)
    if env_file:
        cmd = 'source ' + shlex.quote(str(env_file)) + ' >/dev/null && env -0'
        loaded = subprocess.check_output(['bash', '-c', cmd])
        for item in loaded.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode()
    os.environ['ASCEND_INSTALL_PATH'] = str(CANN_PATH)

HAS_ASCEND = platform.system() == 'Linux' and CANN_PATH is not None and shutil.which('npu-smi') is not None
print('数据文件：', DATA_FILE)
print('实验目录：', WORK_DIR)
print('CANN 路径：', CANN_PATH if CANN_PATH else '未检测到')
print('NPU 环境初检：', '已检测到 CANN 与 npu-smi；构建时继续检查 ASC 工具链和设备' if HAS_ASCEND else '未具备；后续仅运行 CPU 检查')


---
## 2. 任务分析

### 2.1 股票交易评估问题

股票在一个交易日内会以不同价格、不同成交量多次成交。交易者可以将自身成交价与 VWAP 比较，评价成交价相对市场平均水平的优劣。从卖方视角看，在同一统计范围内，成交价高于 VWAP 表示卖出价格优于该基准。

VWAP 以成交量作为权重，其计算公式为：

$$
\mathrm{VWAP}=\frac{\displaystyle\sum_{i=1}^{n}T_i}{\displaystyle\sum_{i=1}^{n}V_i}
$$

其中，$n$ 为统计范围内的成交记录数，$T_i$ 为第 $i$ 笔成交的成交额（美元），$V_i$ 为第 $i$ 笔成交的成交量（股），因此 VWAP 的单位为美元每股。

本实验输入一笔成交的交易编号 `trade_id`，将其成交价与 **同一股票、同一交易日，从首条记录累计至该笔成交完成** 的 VWAP 比较：

- 成交价大于 VWAP，输出“投资成功”；
- 成交价小于或等于 VWAP，输出“投资失败”；
- 交易编号不存在，输出“查找失败”。

**注：** 这里的评价标签只表示成交价格相对 VWAP 的优劣。数据未包含投资者身份和交易方向，本实验统一假定待评价成交来自卖方。


### 2.2 结构体定义

`trades.txt` 首行为表头，字段之间以空白分隔。每条记录包含以下信息：

| 字段 | 含义 |数据类型 | 说明 |
| --- | --- | --- | --- |
| `trade_id` | 交易编号 | `int` | 唯一标识成交记录 |
| `stockcode` | 股票代码 | `string` | 如 AAPL、MSFT |
| `date` | 交易日期 | `string` | YYYYMMDD 格式 |
| `time` | 交易时间 | `long long` | 美东时间当天零点起累计的纳秒数 |
| `volume_t` | 成交量 | `int` | 单位为股 |
| `price` | 成交价 | `int` | 美元每股价格放大 10000 倍后的整数 |
| `turnover_t` | 成交额 | `long long` | 美元成交额放大 10000 倍后的整数 |

根据上述字段，股票交易记录的结构体定义如下：

```cpp
typedef struct trades {
    int trade_id;          // 交易编号
    string stockcode;      // 股票代码
    string date;           // 交易日期
    long long time;        // 交易时间
    int volume_t;          // 成交量
    int price;             // 成交价
    long long turnover_t;  // 成交额
} trades;                 // 股票交易记录
```


### 2.3 实验参数

本实验的参数主要包括：

- `TRADE_ID`：待评价的交易编号；
- `CORE_NUM`：第一阶段启动的 Vector 核数，本实验默认设置为 32；第二阶段固定使用 1 个 Vector 核完成汇总；
- `WARMUP`、`REPEAT`：预热与正式计时次数；
- `SOC_VERSION`：实际设备对应的编译型号，留空时由运行脚本检测。

In [ ]:
# 本次运行参数：第一阶段默认启动 32 个 Vector 核。
TRADE_ID = 265877384      # 本任务的输入，在本讲解设置默认值为265877384
CORE_NUM = 32
WARMUP = 2
REPEAT = 10
SOC_VERSION = 'ascend910b4'

---
## 3. 算法讲解

### 3.1 Host/Device 侧分工

分工：

- **Host侧：** 指 CPU 端，负责文件读取、顺序表构建、交易查找、NPU调度和结果评价；
- **Device侧：** 指 NPU 端，负责成交量与成交额的并行求和。
- **核函数（Kernel）：** 指Device侧 NPU 执行的函数。


数据处理流程：

1. Host 读取文件，构建保存全部成交记录的顺序表；
2. 按交易编号查找目标记录，筛选出本次计算需要的成交记录；
3. 从筛选后的顺序表中提取成交量和成交额，形成两个连续数值数组；
4. 将两个数组传到 Device，由多个计算核分别计算不同区间的成交量合计与成交额合计，得到各核的局部结果；
5. 在 Device 上运行第二个核函数，将各核的成交量合计相加、成交额合计相加，得到整个累计范围的两项总和；
6. Host 取回两项总和，计算 VWAP，并输出交易评估结果。



### 3.2 数据分块

设本次累计范围包含 `n` 条记录，启动 `cores` 个计算核。先求出每核至少处理的记录数 `base = n / cores`，再把余下的 `extra = n % cores` 条记录分给前 `extra` 个核。这样各核处理的数量最多相差一条。

下面以 17 条记录分给 4 个核处理为例，展示数据分块的实现：

| 计算核编号 | 记录数 | 数据块下标（左闭右开） |
| --- | --- | --- |
| 0 | 5 | [0, 5) |
| 1 | 4 | [5, 9) |
| 2 | 4 | [9, 13) |
| 3 | 4 | [13, 17) |

各核在自己的区间内顺序累加，多个核同时处理不同区间。每核只写入自己的局部结果位置，避免多个核修改同一项数据。记录少于计算核数量时，多出的核负责空区间并写入零。

**注：** “计算核”是指 AI Core 上的 Vector 核（AIV），具体代码实现中，核函数通过 `__vector__` 修饰符指定在 Vector 核上执行。因此 `CORE_NUM = 32` 表示启动 32 个 Vector 核，不能直接理解为启动 32 个完整的 AI Core。核内的标量单元负责本实验的整数累加。

### 3.3 并行求和

并行求和分为两个阶段，两个阶段都由 NPU 执行：

1. **局部求和：** 32 个计算核同时处理各自负责的数据块。每个核分别得到成交量合计和成交额合计，因此第一阶段结束后共有 32 组局部结果。
2. **结果汇总：** 由 1 个计算核把这 32 组成交量合计相加、成交额合计相加，得到整个累计范围的成交量总和与成交额总和。

**示例：** 假设现共有 16000 条记录，则16 个计算核各处理 1000 条。

- 第一阶段：16 个计算核各自得到一组成交量合计和成交额合计，共 16 组局部结果，每组包含两个数；
- 第二阶段：分别将这 16 组结果中的成交量合计相加、成交额合计相加，得到成交量总和与成交额总和，再将这两个总和返回给 CPU。

本实验的求和就是整数累加，每个核使用循环，逐条读取数值并相加。多个核同时对不同区间执行这样的循环，就形成核之间的数据并行。本实验重点学习各核的分工与结果汇总。

**注：** Host 侧的成交量合计和成交额合计均使用 `long long` 64 位整数类型；Device 侧使用对应的 64 位整数类型 `int64_t`。


---
## 4. 代码构建与运行

本节按照实验工程的目录结构组织代码，依次完成**代码实现 → 项目构建 → 运行并输出实验结果**。

#### 项目结构

**提示：** 点击 **文件名** 可跳转到对应文件的说明及代码单元格。

| 位置 | 文件 | 功能 |
| --- | --- | --- |
| `include/` | [`trades.hpp`](#seq-types) | 成交记录、顺序表与求和结果定义 |
|  | [`seq_list.hpp`](#seq-list-interface) | 顺序表基本操作接口 |
|  | [`trade_utils.hpp`](#seq-trade-interface) | 数据读取、交易评估与实验参数接口 |
| `src/` | [`src/seq_list.cpp`](#seq-list-implementation) | 初始化、构建、插入、查找与销毁 |
|  | [`trade_utils.cpp`](#seq-trade-implementation) | 文件读取、累计范围构建、CPU 求和与评估 |
|  | [`cpu_main.cpp`](#seq-cpu-main) | CPU 运行入口 |
|  | [`test_common.cpp`](#seq-common-test) | 顺序表和算法检查 |
| `ascend/` | [`stock_vwap_npu.asc`](#seq-asc) | Device 侧核函数、Host 侧设备管理、数据传输与调用 |
| `scripts/` | [`run.sh`](#seq-run-script) | CANN 环境加载、构建与运行 |
| 项目根目录 | [`CMakeLists.txt`](#seq-cmake) | 项目编译配置 |

**注：** `results` 中的文件由运行过程生成。

| 文件 | 功能 |
| --- | --- |
| [工程检查与 CPU 编译](#seq-build-check) | 检查文件并运行 CPU 测试 |
| [CPU 运行单元格](#seq-run-cpu) | 生成 CPU 参考结果 |
| [NPU 运行单元格](#seq-run-npu) | 生成 NPU 结果及各核检查信息 |

### 4.0 实操任务

本实践围绕股票交易评估，依次完成顺序表的定义、顺序表的构建、数据分块、Ascend C 并行求和及 VWAP 交易评估，理解如何组织成交记录，并将求和工作分配给多个 NPU 计算核共同完成。

完成后应能够：

1. 定义存储股票交易记录的顺序表，理解存储空间基地址与当前表长的含义；
2. 通过初始化和循环存入记录构建顺序表，结合顺序查找确定交易评估所需的累计范围；
3. 根据记录数、计算核数量和核编号划分连续数据块，使各核处理的记录不重复、不遗漏，且数量尽可能均衡；
4. 使用 Ascend C 实现各核对成交量与成交额的局部求和，结合已经给出的汇总代码，理解多个计算核怎样共同完成求和；
5. 根据返回的成交量合计与成交额合计计算 VWAP，完成卖方视角的股票交易评估，并核对 CPU 与 NPU 的结果。

操作方式：

1. 先阅读任务说明；
2. 在带有 `TODO` 的代码单元中补全代码；
3. 完成补全并生成对应文件后，运行第四部分末尾的检查单元，检查该任务是否正确：通过时显示 `PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。
4. 测试通过后再进入下一项任务。

本实践共包含 5 个任务：

- **任务一：** 顺序表的定义——使用标准 C++ 定义顺序表
- **任务二：** 顺序表的构建——基于顺序表组织股票交易记录
- **任务三：** 数据分块——基于顺序表为 NPU 各核划分数据块
- **任务四：** 基于 Ascend C 实现并行求和——在昇腾 NPU 上分别对成交量与成交额求和
- **任务五：** VWAP 计算与交易评估——根据累计 VWAP 评价卖出成交价格

#### 4.0.1 任务一：顺序表的定义

根据已经给出的股票交易记录结构体 `trades`，定义顺序表 `SqList`。理解一条成交记录与保存多条记录的顺序表之间的关系，以及当前表长与最大容量的区别。

请补全：

- `TODO 1A`：在 `SqList` 中补全两条成员定义语句，使用 `trades *elem` 保存成交记录存储空间的基地址，使用 `int length` 保存当前表长。（[4.1.1（1）成交记录与顺序表：`trades.hpp`](#seq-types-code)）

**提示：** 完成后运行[任务一检查](#task1-check)，检查成员名称、类型及记录访问是否正确。

#### 4.0.2 任务二：顺序表的构建

实现顺序表的初始化与构建。文件读取代码已经给出，先将 `trades.txt` 中的数据解析为 `trades` 类型的记录数组，再用一个 `for` 循环将各条记录按原顺序存入顺序表。

请补全：

- `TODO 2A`：补全 `InitList_Sq`，申请可存放 `MAXSIZE` 条成交记录的数组空间，并将表长置为 0，建立空顺序表。（[4.1.2（1）顺序表基本操作：`seq_list.cpp`](#seq-list-code)）
- `TODO 2B`：补全 `CreateList_Sq`，用一个 `for` 循环将记录数组 `records` 中的 `n` 条成交记录依次存入顺序表，每存入一条记录将表长加 1。（[4.1.2（1）顺序表基本操作：`seq_list.cpp`](#seq-list-code)）

**提示：** 完成后运行[任务二检查](#task2-check)，分别检查初始化和顺序表构建操作。

#### 4.0.3 任务三：数据分块

使用辅助代码从顺序表中提取的成交量与成交额数组，根据记录总数与计算核数量，计算每核至少处理的记录数，以及需要分配的剩余记录数。后续代码结合当前核编号，确定各核的起始下标和结束下标。两列数组采用相同规则，使同一笔成交的两个数值由同一个核处理。

分块时应保证所有记录均被处理一次，各核处理的记录数最多相差一条；当记录数少于计算核数量时，允许部分核处理空区间。本实验第一阶段默认启动 32 个 Vector 核。

请补全：

- `TODO 3A`：用整数除法计算每核至少处理的记录数 `base`，用取余运算计算剩余记录数 `extra`。其余区间计算代码已经给出。（[4.1.3（1）多核局部求和：`stock_vwap_npu.asc`](#seq-partial-code)）

**提示：** 完成后运行[任务三检查](#task3-check)，检查整除、有余数以及记录数少于核数时的分配结果。

#### 4.0.4 任务四：基于 Ascend C 实现并行求和

在给出的 Ascend C 代码框架中，实现各核对所负责区间的成交量与成交额分别累加。多个核同时处理不同数据块，每核独立保存两项局部合计。

局部结果写回、设备管理、数据传输及核函数启动代码已经给出；第二阶段的单个核会分别汇总各核的局部结果，得到整个累计范围的成交量总和与成交额总和。通过实验体会“划分数据—各核计算—汇总结果”的并行处理过程。

请补全：

- `TODO 4A`：使用一个 `for` 循环，遍历当前核的 `[begin, end)` 区间，调用 `GetValue(i)` 读取成交量和成交额，分别累加到 `volumeSum` 与 `turnoverSum`。（[4.1.3（1）多核局部求和：`stock_vwap_npu.asc`](#seq-partial-code)）

**提示：** 完成后运行[任务四检查](#task4-check)，检查局部累加逻辑；后续运行 NPU 程序，核对各核局部结果和两项总和。

#### 4.0.5 任务五：VWAP 计算与交易评估

使用求和结果计算同一股票当日截至目标成交（含）的累计 VWAP。成交额按 0.0001 美元存储，成交价按 0.0001 美元/股存储，先使用已给出的代码换算为美元和美元/股，再进行计算与比较。

本实验统一从卖方视角评价成交价格：目标成交价高于累计 VWAP 时返回“投资成功”，否则返回“投资失败”；成交价与 VWAP 相等时也返回“投资失败”。主程序负责输出返回的评价结果。

请补全：

- `TODO 5A`：用累计成交额 `turnover` 除以累计成交量 `volume` 得到 `vwap`，随后使用 `if/else` 比较 `price` 与 `vwap`，返回对应的评价结果。（[4.1.2（2）数据读取、累计范围与 CPU 算法：`trade_utils.cpp`](#seq-trade-code)）

**提示：** 完成后运行[任务五检查](#task5-check)，检查成交价高于、等于和低于 VWAP 时的评价结果。


### 4.1 代码实现

<a id="seq-types"></a>

#### 4.1.1 `include`：股票记录、顺序表及公共接口


##### （1）成交记录与顺序表：`trades.hpp`

股票交易记录使用第 2.2 节中给出的结构体的定义。

顺序表 `SqList` 只包含两个成员：
- `*elem` 指向连续的成交记录存储空间；
- `length` 表示当前表长。

最大容量由 `seq_list.hpp` 中的常量 `MAXSIZE` 统一规定，本实验设置为 3000000。

**注：** 数组下标从 0 开始，第 i 个元素对应 `elem[i - 1]`。

**任务一 TODO：**

- `TODO 1A`：在 `SqList` 中补全两条成员定义语句，使用 `trades *elem` 保存成交记录存储空间的基地址，使用 `int length` 保存当前表长。

<a id="seq-types-code"></a>

In [ ]:
%%writefile Source/stock_trading_evaluation_seq/include/trades.hpp
#ifndef STOCK_TRADES_HPP
#define STOCK_TRADES_HPP
#include <string>
using namespace std;

// 一条记录对应一笔成交，价格和成交额均保存为放大后的整数。
typedef struct trades {
    int trade_id;          // 交易编号
    string stockcode;      // 股票代码
    string date;           // 交易日期
    long long time;        // 交易时间：当天零点起的纳秒数
    int volume_t;          // 成交量：股
    int price;             // 成交价：0.0001 美元/股
    long long turnover_t;  // 成交额：0.0001 美元
} trades;

typedef struct {
    // TODO 1A：定义存储空间基地址 elem 和当前表长 length 两个成员。
    trades *elem;  // 存储空间的基地址
    int length;    // 当前长度
    // END TODO 1A
} SqList;

// NPU 输入只保留计算需要的两列整数，两个数组下标一一对应。
typedef struct {
    long long *volume;     // 提取的成交量，提升为 64 位整数
    long long *turnover;   // 提取的成交额
    int length;
} NumericInput;

// CPU 与 NPU 使用相同的两项求和结果。
typedef struct {
    long long volume;      // 成交量合计
    long long turnover;    // 成交额合计
} SumResult;

// 每核写入四个 64 位整数，共 32 字节。
// 前两项为局部合计，后两项为区间起点与终点，供正确性检查使用。
enum { PARTIAL_STRIDE = 4, MAX_CORE_NUM = 32 };
#endif


<a id="seq-list-interface"></a>
##### （2）顺序表基本操作接口：`seq_list.hpp`

按照“定义结构—实现基本操作—使用基本操作”的方式组织顺序表代码。`InitList_Sq` 完成初始化，`CreateList_Sq` 根据记录数组构建顺序表，两者成功时返回 `true`，不能完成操作时返回 `false`。插入、查找和内存释放接口作为辅助代码提供，查找成功返回逻辑位序，失败返回 0。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/include/seq_list.hpp
#ifndef STOCK_SEQ_LIST_HPP
#define STOCK_SEQ_LIST_HPP
#include "trades.hpp"
using namespace std;

const int MAXSIZE = 3000000;  // 顺序表的最大容量

// 初始化空表、按原顺序构建表，以及销毁表。
bool InitList_Sq(SqList &L);
bool CreateList_Sq(SqList &L, const trades records[], int n);
void DestroyList_Sq(SqList &L);
// 辅助操作：插入使用从 1 开始的位序，查找失败返回 0。
bool ListInsert_Sq(SqList &L, int i, trades e);
int LocateTrade_Sq(const SqList &L, int tradeId);
#endif


<a id="seq-trade-interface"></a>
##### （3）交易处理与实验参数接口：`trade_utils.hpp`

本文件声明文件读取、累计范围构建、数值提取、求和和交易评估接口。`Options` 保存运行参数，结果记录接口用于生成第 4 部分读取的 CSV 文件。文件解析、参数检查及计时属于辅助代码。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/include/trade_utils.hpp
#ifndef STOCK_TRADE_UTILS_HPP
#define STOCK_TRADE_UTILS_HPP
#include "seq_list.hpp"
#include <string>
using namespace std;

// 数据处理接口：读取、筛选累计范围和提取数值数组。
void LoadTrades(const string &path, SqList &list);
bool BuildPrefix(const SqList &all, int tradeId, SqList &prefix);
void ExtractNumeric(const SqList &prefix, NumericInput &input);
void DestroyNumeric(NumericInput &input);
// CPU 求和作为参照；评价函数可直接使用 NPU 返回的 SumResult。
SumResult SumCpu(const NumericInput &input);
double CalcVwap(const SumResult &sum);
string EvaluateTrade(const trades &target, const SumResult &sum);
void GetRange(int n, int cores, int core, int &begin, int &end);
double NowUs();

// 命令行参数集中保存在此结构中；下方解析代码作为辅助代码提供。
struct Options {
    string dataPath;
    string csvPath;
    int tradeId;
    int cores;
    int device;
    int warmup;
    int repeat;
    bool hasTradeId;
    bool diagnostic;
    Options();
};

// 运行参数、CPU 计时和结果文件输出接口。
Options ParseArgs(int argc, char **argv);
double BenchmarkCpu(const NumericInput &input, int warmup, int repeat,
                    SumResult &result);
void WriteMetrics(const string &path, const char *implementation,
                  const Options &opt, int n, int cores, int activeCores,
                  const SumResult &sum, const string &decision,
                  double avgUs, bool pass);
#endif


#### 4.1.2 `src`：数据读取、CPU 参考算法和测试程序

先实现顺序表的基本操作，再使用这些操作组织成交记录。输入输出、数据处理和测试程序分别存放，便于按功能阅读。


<a id="seq-list-implementation"></a>

##### （1）顺序表基本操作：`seq_list.cpp`

`InitList_Sq` 按最大容量 `MAXSIZE` 分配存储空间并建立空表。`CreateList_Sq` 接收已经初始化的空顺序表、成交记录数组 `records` 和记录数 `n`，通过一个 `for` 循环按原顺序保存记录，并更新表长。

插入、查找和内存释放函数已经提供。`ListInsert_Sq` 用于辅助代码中的记录插入，`LocateTrade_Sq` 按交易编号进行顺序查找。成交记录中包含 `string` 对象，因此使用 `new[]` 与 `delete[]` 申请和释放数组。

**任务二 TODO：**

- `TODO 2A`：申请可存放 `MAXSIZE` 条成交记录的数组空间，并将表长置为 0，建立空顺序表。
- `TODO 2B`：补全 `CreateList_Sq`，用一个 `for` 循环将记录数组 `records` 中的 `n` 条成交记录依次存入顺序表，每存入一条记录将表长加 1。

<a id="seq-list-code"></a>


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/src/seq_list.cpp
#include "seq_list.hpp"
using namespace std;

// 初始化时申请连续存储空间，当前有效元素数从 0 开始。
bool InitList_Sq(SqList &L)
{
    if (L.elem != 0) return false;

    // TODO 2A：申请 MAXSIZE 条成交记录的空间，并将表长置为 0。
    L.elem = new trades[MAXSIZE];
    L.length = 0;
    // END TODO 2A

    return true;
}

bool CreateList_Sq(SqList &L, const trades records[], int n)
{
    // 构建操作以已初始化的空表为起点，容量与参数检查已经给出。
    if (L.elem == 0 || L.length != 0 || n < 0 || n > MAXSIZE) return false;
    if (n > 0 && records == 0) return false;

    // TODO 2B：用一个 for 循环依次存入各条成交记录，并更新表长。
    for (int i = 0; i < n; ++i) {
        L.elem[L.length] = records[i];
        ++L.length;
    }
    // END TODO 2B
    
    return true;
}

// new[] 对应 delete[]，释放后重置指针与表长。
void DestroyList_Sq(SqList &L)
{
    delete [] L.elem;
    L.elem = 0;
    L.length = 0;
}

bool ListInsert_Sq(SqList &L, int i, trades e)
{
    if (i < 1 || i > L.length + 1) return false;
    if (L.length >= MAXSIZE) return false;
    // 辅助插入操作已经提供；第 i 个位置对应数组下标 i-1。
    // 从后向前移动元素，避免覆盖尚未移动的记录。
    for (int j = L.length - 1; j >= i - 1; j--)
        L.elem[j + 1] = L.elem[j];
    L.elem[i - 1] = e;
    ++L.length;
    return true;
}

// 顺序查找：找到时返回逻辑位序，未找到返回 0。
int LocateTrade_Sq(const SqList &L, int tradeId)
{
    for (int i = 0; i < L.length; ++i)
        if (L.elem[i].trade_id == tradeId) return i + 1;
    return 0;
}


<a id="seq-trade-implementation"></a>
##### （2）数据读取、累计范围与 CPU 算法：`trade_utils.cpp`

读取文件时先统计记录数并检查容量，再将各行解析为 `trades` 类型的临时记录数组。初始化顺序表后，调用 `CreateList_Sq` 执行 TODO 2B 中的循环，将各条记录按原顺序存入顺序表；构建完成后释放临时数组。

`BuildPrefix` 查找目标成交的位序，只遍历文件开头至该目标记录，筛选股票代码和交易日期均相同的记录。这样构建出的顺序表以目标成交结尾，目标之后的同时间成交也不会被计入。

`ExtractNumeric` 从记录中提取两列连续数值，`SumCpu` 分别顺序累加，作为 NPU 的校验依据。`EvaluateTrade` 将累计成交额换算为美元、成交价换算为美元/股，再计算 VWAP 并完成卖方评价。`CalcVwap` 是结果记录使用的辅助函数，已经完整提供。

**任务五 TODO：**

- `TODO 5A`：在 `EvaluateTrade` 中计算 `vwap = turnover / volume`，随后使用 `if/else` 判断成交价是否高于 VWAP，返回“投资成功”或“投资失败”。单位换算和成交量检查已经给出。

<a id="seq-trade-code"></a>


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/src/trade_utils.cpp
#include "trade_utils.hpp"
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>
#include <stdexcept>
#include <climits>
#include <ctime>
using namespace std;

// 识别空白行，文件解析时跳过这些行。
static bool IsBlank(const string &line)
{
    return line.find_first_not_of(" \t\r\n") == string::npos;
}

// 先检查表头和容量，再解析记录，最后调用顺序表构建操作。
void LoadTrades(const string &path, SqList &list)
{
    ifstream in(path.c_str());
    if (!in) throw runtime_error("cannot open trades.txt: " + path);
    string line, token, extra;
    if (!getline(in, line)) throw runtime_error("empty data file");
    const char *names[] = {"trade_id", "stockcode", "date", "time",
                           "volume_t", "price", "turnover_t"};
    istringstream header(line);
    for (int i = 0; i < 7; ++i) {
        if (!(header >> token) || token != names[i])
            throw runtime_error("unexpected header in trades.txt");
    }
    if (header >> extra) throw runtime_error("unexpected header field");

    int count = 0;
    while (getline(in, line)) {
        if (!IsBlank(line)) {
            if (count == INT_MAX) throw runtime_error("too many records");
            ++count;
        }
    }
    if (count == 0) throw runtime_error("no trade records");
    if (count > MAXSIZE) throw runtime_error("trade records exceed MAXSIZE");
    if (!InitList_Sq(list))
        throw runtime_error("cannot initialize trade list");
    // 临时数组只用于文件解析，构建完成后立即释放。
    trades *records = new trades[count];
    try {
        in.clear();
        in.seekg(0);
        getline(in, line);  // 跳过表头
        int lineNumber = 1;
        int loaded = 0;
        while (getline(in, line)) {
            ++lineNumber;
            if (IsBlank(line)) continue;
            trades record;
            istringstream row(line);
            if (!(row >> record.trade_id >> record.stockcode >> record.date
                      >> record.time >> record.volume_t >> record.price
                      >> record.turnover_t) || (row >> extra)) {
                ostringstream message;
                message << "invalid trade at line " << lineNumber;
                throw runtime_error(message.str());
            }
            // 先提升乘数类型，避免两个 int 的中间乘积溢出。
            long long expected = (long long)(record.volume_t) * record.price;
            if (record.trade_id <= 0 || record.date.size() != 8 ||
                record.time < 0 || record.time >= 86400000000000LL ||
                record.volume_t <= 0 || record.price <= 0 ||
                record.turnover_t != expected)
                throw runtime_error("invalid trade field value");
            if (loaded >= count) throw runtime_error("data file changed while reading");
            records[loaded] = record;
            ++loaded;
        }
        if (loaded != count) throw runtime_error("data file changed while reading");
        // 将解析好的记录数组交给顺序表构建函数，正式运行会执行 TODO 2B。
        if (!CreateList_Sq(list, records, count))
            throw runtime_error("cannot build trade list");
    } catch (...) {
        delete [] records;
        throw;
    }
    delete [] records;
}

// 按目标记录的位置截取，目标之后即使时间相同也不纳入统计。
bool BuildPrefix(const SqList &all, int tradeId, SqList &prefix)
{
    int position = LocateTrade_Sq(all, tradeId);  // 返回从 1 开始的逻辑位序
    if (position == 0) return false;
    const trades &target = all.elem[position - 1];
    if (!InitList_Sq(prefix))
        throw runtime_error("cannot initialize prefix list");
    for (int i = 0; i < position; ++i) {
        if (all.elem[i].stockcode == target.stockcode &&
            all.elem[i].date == target.date) {
            if (!ListInsert_Sq(prefix, prefix.length + 1, all.elem[i]))
                throw runtime_error("cannot append prefix record");
        }
    }
    return true;
}

// 只把成交量与成交额提取为连续数组，供 CPU 与 NPU 共用。
void ExtractNumeric(const SqList &prefix, NumericInput &input)
{
    // 两个数组的相同下标对应同一笔成交，不传送股票代码和日期字符串。
    input.length = prefix.length;
    input.volume = new long long[input.length];
    input.turnover = new long long[input.length];
    for (int i = 0; i < input.length; ++i) {
        input.volume[i] = prefix.elem[i].volume_t;
        input.turnover[i] = prefix.elem[i].turnover_t;
    }
}

void DestroyNumeric(NumericInput &input)
{
    delete [] input.volume;
    delete [] input.turnover;
    input.volume = 0;
    input.turnover = 0;
    input.length = 0;
}

// CPU 逐项求和，为检查 NPU 计算结果提供独立参照。
SumResult SumCpu(const NumericInput &input)
{
    SumResult sum = {0, 0};  // 两项累计值均为 long long

    for (int i = 0; i < input.length; ++i) {
        sum.volume += input.volume[i];
        sum.turnover += input.turnover[i];
    }
    return sum;
}

double CalcVwap(const SumResult &sum)
{
    if (sum.volume <= 0) throw runtime_error("invalid total volume");
    // 成交额按 0.0001 美元存储，计算结果换算为美元/股。
    return ((double)(sum.turnover) / sum.volume) / 10000.0;
}

// 根据累计成交量和成交额，从卖方视角评价目标成交价。
string EvaluateTrade(const trades &target, const SumResult &sum)
{
    double price = target.price / 10000.0;  // 目标成交价：美元/股
    double vwap = 0;
    if (sum.volume <= 0)
        throw runtime_error("invalid total volume");
    double turnover = sum.turnover / 10000.0;  // 累计成交额：美元
    long long volume = sum.volume;            // 累计成交量：股

    // TODO 5A：计算 VWAP，并根据卖方评价规则返回结果。
    vwap = turnover / volume;
    if (price > vwap)
        return "投资成功";
    else
        return "投资失败";
    // END TODO 5A
        
}

// CPU 参考分块规则，用于核对 NPU 各核实际处理的区间。
void GetRange(int n, int cores, int core, int &begin, int &end)
{
    int base = n / cores;
    int extra = n % cores;
    // 当前核之前共有 precedingExtra 条额外记录。
    int precedingExtra = extra;
    if (core < extra)
        precedingExtra = core;
    begin = core * base + precedingExtra;
    end = begin + base;
    if (core < extra)
        ++end;
}

// 使用单调时钟取得微秒时间，供平均耗时统计使用。
double NowUs()
{
    struct timespec t;
    if (clock_gettime(CLOCK_MONOTONIC, &t) != 0)
        throw runtime_error("clock_gettime failed");
    return (double)(t.tv_sec) * 1000000.0 + t.tv_nsec / 1000.0;
}

// 默认第一阶段使用 32 个计算核，第二阶段固定使用一个核。
Options::Options() : dataPath("trades.txt"), tradeId(0), cores(32), device(0),
                     warmup(2), repeat(10), hasTradeId(false), diagnostic(false) {}

// 以下为命令行解析辅助代码，不属于学生需要补全的算法。
static int ParseInt(const string &text)
{
    istringstream in(text);
    int value;
    char extra;
    if (!(in >> value) || (in >> extra)) throw runtime_error("invalid integer argument");
    return value;
}

Options ParseArgs(int argc, char **argv)
{
    Options opt;
    for (int i = 1; i < argc; ++i) {
        string name(argv[i]);
        if (name == "--diagnostic") { opt.diagnostic = true; continue; }
        if (i + 1 >= argc) throw runtime_error("missing argument value: " + name);
        string value(argv[++i]);
        if (name == "--data") opt.dataPath = value;
        else if (name == "--csv") opt.csvPath = value;
        else if (name == "--trade-id") { opt.tradeId = ParseInt(value); opt.hasTradeId = true; }
        else if (name == "--cores") opt.cores = ParseInt(value);
        else if (name == "--device") opt.device = ParseInt(value);
        else if (name == "--warmup") opt.warmup = ParseInt(value);
        else if (name == "--repeat") opt.repeat = ParseInt(value);
        else throw runtime_error("unknown argument: " + name);
    }
    if (opt.cores < 1 || opt.cores > MAX_CORE_NUM || opt.device < 0 ||
        opt.warmup < 0 || opt.repeat < 1)
        throw runtime_error("invalid cores, device, warmup or repeat");
    // 未使用命令行指定编号时，从标准输入读取，不额外输出提示。
    if (!opt.hasTradeId && !(cin >> opt.tradeId))
        throw runtime_error("please input a trade_id");
    return opt;
}

void WriteMetrics(const string &path, const char *implementation,
                  const Options &opt, int n, int cores, int activeCores,
                  const SumResult &sum, const string &decision,
                  double avgUs, bool pass)
{
    if (path.empty()) return;
    ofstream out(path.c_str());
    if (!out) throw runtime_error("cannot write result CSV: " + path);
    // CSV 保存实验记录；业务程序的标准输出仍只有评价标签。
    string status;
    if (pass)
        status = "PASS";
    else
        status = "FAIL";
    out << "implementation,trade_id,n,cores,active_cores,volume_sum,turnover_sum,vwap,decision,avg_us,status\n";
    out << implementation << ',' << opt.tradeId << ',' << n << ',' << cores << ','
        << activeCores << ',' << sum.volume << ',' << sum.turnover << ','
        << fixed << setprecision(8) << CalcVwap(sum) << ',' << decision << ','
        << setprecision(3) << avgUs << ',' << status << '\n';
}


<a id="seq-cpu-main"></a>
##### （3）CPU 运行入口：`cpu_main.cpp`

主程序按照“读取数据—构建累计范围—提取数值—顺序求和—输出评估”的顺序执行。正常标准输出只有评估标签，辅助信息写入标准错误流，详细实验结果保存在 CSV 中。

计时循环与求和函数放在不同源文件中，使用常规非 LTO 编译，避免同一编译单元的优化将重复求和折叠为一次计算。

本文件作为辅助代码完整提供，无需补全。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/src/cpu_main.cpp
#include "trade_utils.hpp"
#include <iostream>
#include <exception>
using namespace std;

// 计时循环与 SumCpu 分处不同源文件，常规非 LTO 编译下逐次调用求和。
double BenchmarkCpu(const NumericInput &input, int warmup, int repeat, SumResult &result)
{
    // 预热不计时，正式重复运行后取平均值。
    for (int i = 0; i < warmup; ++i) result = SumCpu(input);
    double start = NowUs();
    for (int i = 0; i < repeat; ++i) result = SumCpu(input);
    return (NowUs() - start) / repeat;
}

// 按读取、筛选、数值提取、求和、评价的顺序执行，并在结束前释放空间。
int main(int argc, char **argv)
{
    SqList all = {0, 0};
    SqList prefix = {0, 0};
    NumericInput input = {0, 0, 0};
    int exitCode = 0;
    try {
        Options opt = ParseArgs(argc, argv);
        LoadTrades(opt.dataPath, all);
        if (!BuildPrefix(all, opt.tradeId, prefix)) {
            cout << "查找失败\n";
        } else {
            ExtractNumeric(prefix, input);
            SumResult sum = {0, 0};
            double avgUs = BenchmarkCpu(input, opt.warmup, opt.repeat, sum);
            string decision = EvaluateTrade(prefix.elem[prefix.length - 1], sum);
            WriteMetrics(opt.csvPath, "cpu", opt, input.length, 1, 1, sum,
                         decision, avgUs, true);
            if (opt.diagnostic)
                cerr << "[CPU] loaded=" << all.length << " prefix=" << input.length
                          << " volume=" << sum.volume << " turnover=" << sum.turnover << '\n';
            cout << decision << '\n';
        }
    } catch (const exception &error) {
        cerr << "[ERROR] " << error.what() << '\n';
        exitCode = 1;
    }
    DestroyNumeric(input);
    DestroyList_Sq(prefix);
    DestroyList_Sq(all);
    return exitCode;
}


<a id="seq-common-test"></a>
##### （4）顺序表与算法检查：`test_common.cpp`

测试使用人工记录检查顺序表的初始化与构建，以及辅助插入函数的元素移动、非法位序、容量限制、查找位序、同股票同日期筛选和目标记录边界。随后检查成交量与成交额求和、价格相等时的规则，以及不同记录数和核数下的区间覆盖。

人工记录只用于检查基本操作，正式股票评估仍读取同目录下的 `trades.txt`。

本文件作为辅助代码完整提供，无需补全。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/src/test_common.cpp
#include "trade_utils.hpp"
#include <iostream>
#include <stdexcept>
using namespace std;

// 小规模人工样例用于检查基本操作和边界，不替代真实数据文件。
static void Check(bool condition, const char *message)
{
    if (!condition) throw runtime_error(message);
}

// 构造一条完整的人工成交记录，乘法前先转为 long long。
static trades MakeTrade(int id, const char *stock, const char *date, int volume, int price)
{
    trades t;
    t.trade_id = id; t.stockcode = stock; t.date = date;
    t.time = 34200000000000LL;
    t.volume_t = volume; t.price = price;
    t.turnover_t = (long long)(volume) * price;
    return t;
}

int main()
{
    SqList L = {0, 0}, all = {0, 0}, prefix = {0, 0};
    NumericInput input = {0, 0, 0};
    try {
        Check(sizeof(long long) == 8, "long long must be 64-bit");
        Check(InitList_Sq(L) && L.length == 0, "initialization failed");
        Check(ListInsert_Sq(L, 1, MakeTrade(1, "AAPL", "20191230", 1, 100)), "insert failed");
        Check(ListInsert_Sq(L, 2, MakeTrade(3, "AAPL", "20191230", 1, 100)), "append failed");
        Check(ListInsert_Sq(L, 2, MakeTrade(2, "AAPL", "20191230", 1, 100)), "middle insertion failed");
        Check(L.elem[0].trade_id == 1 && L.elem[1].trade_id == 2 && L.elem[2].trade_id == 3,
              "element movement failed");
        // 按值传递待插入元素，移动数组时仍能正确插入表内某元素的副本。
        Check(ListInsert_Sq(L, 1, L.elem[2]) && L.elem[0].trade_id == 3 &&
              L.elem[2].trade_id == 2, "insertion of an existing element failed");
        Check(!ListInsert_Sq(L, 0, L.elem[0]), "invalid position accepted");
        Check(LocateTrade_Sq(L, 2) == 3 && LocateTrade_Sq(L, 999) == 0, "lookup failed");

        // 顺序表存满 MAXSIZE 条记录后，应拒绝继续插入。
        while (L.length < MAXSIZE)
            Check(ListInsert_Sq(L, L.length + 1, L.elem[0]), "fill list failed");
        Check(!ListInsert_Sq(L, MAXSIZE + 1, L.elem[0]) &&
              L.length == MAXSIZE, "capacity check failed");

        // 同一时间的成交仍按记录位置确定累计边界。
        Check(InitList_Sq(all), "initialization failed");
        trades samples[5];
        samples[0] = MakeTrade(1, "AAPL", "20191230", 2, 1000000);
        samples[1] = MakeTrade(2, "MSFT", "20191230", 3, 2000000);
        samples[2] = MakeTrade(3, "AAPL", "20191231", 9, 9000000);
        samples[3] = MakeTrade(4, "AAPL", "20191230", 1, 1200000);
        samples[4] = MakeTrade(5, "AAPL", "20191230", 7, 8000000);
        Check(CreateList_Sq(all, samples, 5), "build list failed");
        Check(BuildPrefix(all, 4, prefix), "prefix not found");
        Check(prefix.length == 2 && prefix.elem[1].trade_id == 4, "scope or tie boundary failed");
        ExtractNumeric(prefix, input);
        SumResult sum = SumCpu(input);
        Check(sum.volume == 3 && sum.turnover == 3200000, "CPU sum failed");
        Check(EvaluateTrade(prefix.elem[1], sum) == "投资成功", "seller evaluation failed");
        SumResult first = {2, 2000000};
        Check(EvaluateTrade(all.elem[0], first) == "投资失败", "equality rule failed");

        // 包含 n<cores、不能整除以及空区间等情形。
        for (int n = 1; n <= 35; ++n) {
            for (int cores = 1; cores <= MAX_CORE_NUM; ++cores) {
                int previousEnd = 0, total = 0;
                for (int c = 0; c < cores; ++c) {
                    int begin, end;
                    GetRange(n, cores, c, begin, end);
                    Check(begin == previousEnd && begin <= end && end <= n, "partition failed");
                    total += end - begin;
                    previousEnd = end;
                }
                Check(total == n && previousEnd == n, "coverage failed");
            }
        }
        cout << "CPU list / scope / sum / partition checks: PASS\n";
    } catch (const exception &e) {
        DestroyNumeric(input);
        DestroyList_Sq(prefix); DestroyList_Sq(all); DestroyList_Sq(L);
        cerr << e.what() << '\n';
        return 1;
    }
    DestroyNumeric(input);
    DestroyList_Sq(prefix); DestroyList_Sq(all); DestroyList_Sq(L);
    return 0;
}


<a id="seq-asc"></a>
#### 4.1.3 Ascend C 程序：`ascend/stock_vwap_npu.asc`

Host 与 Device 写在同一个 `.asc` 源文件中，由毕昇编译器分别生成两侧的程序。普通函数在 Host 上负责准备数据和调度；带有执行修饰符的核函数在 NPU 上计算。二者仍通过设备内存地址和执行流配合。

阅读代码前，先了解以下写法：

| 写法 | 在本实验中的含义 |
| --- | --- |
| `#include "kernel_operator.h"` | 引入 Ascend C 的核函数接口与设备侧类型 |
| `using namespace AscendC;` | 使设备代码可以直接使用 `GlobalTensor`、`GetBlockIdx()` 等名称 |
| `__global__ __vector__` | 声明可由 Host 启动、在 NPU Vector 核上执行的核函数 |
| `GM_ADDR`、`__gm__` | 表示设备全局内存地址及地址空间，不能在 Host 上直接解引用设备地址 |
| `uint32_t`、`int64_t` | 标准整数类型，分别为 32 位无符号整数和 64 位有符号整数 |
| `GetBlockIdx()`、`GetBlockNum()` | 取得当前计算块编号和本次启动的计算块数，本实验按核分工使用 |
| `InitSocState()` | 初始化本次核函数执行所需的设备状态 |
| `kernel<<<cores, 0, stream>>>(...)` | 由 Host 启动核函数，依次指定计算块数、保留参数（填 0）及执行流；圆括号中传入核函数参数 |
| `#include <acl/acl.h>` | 引入 CANN Runtime 的设备管理、内存传输与执行流接口 |
| `ACL_CHECK(...)` | 本工程定义的错误检查宏，用于检查 Runtime 接口返回值 |
| `aclrtSynchronizeStream(stream)` | Host 等待指定执行流上的任务完成，再读取结果或启动下一步计算 |

`<<<>>>` 启动语句直接提交核函数，不具有 Runtime 接口的返回值形式；执行结束后通过 `aclrtSynchronizeStream` 检查完成状态。`aclrtMalloc` 申请设备内存，`aclrtMemcpy` 在 Host 与 Device 之间传输数据；使用结束后由 `aclrtFree` 等接口释放资源。

<a id="seq-partial-kernel"></a>
<a id="seq-partial-code"></a>
##### （1）多核局部求和：`stock_partial_sum`

各核按相同公式确定互不重叠的 `[begin, end)` 区间，分别累加成交量与成交额。`TODO 3A` 求出每核至少处理的记录数 `base` 和剩余记录数 `extra`；`TODO 4A` 遍历当前核的区间，调用 `GetValue(i)` 完成两项累加。区间计算、结果写回与同步代码已经给出。

`GlobalTensor` 关联 Device 全局内存，`LocalTensor` 访问核内局部存储；`TPipe`、`TBuf` 管理局部缓冲区。每核先在局部缓冲区写入 4 个整数，再用 `DataCopy` 搬出，占用独立的 32 字节结果槽。`SetFlag` 与 `WaitFlag` 保证标量写入完成后再搬运，`PipeBarrier` 等待核内流水操作完成。缓存维护接口用于避免标量读取重复运行时的旧输入。

<a id="seq-merge-kernel"></a>
##### （2）局部结果汇总：`stock_merge_sum`

只启动一个 Vector 核，将各核局部结果整体搬入核内存储。等待搬入完成后，分别汇总各核的成交量与成交额，写回最终结果。各槽的区间边界仅用于检查，不参与求和。

<a id="seq-host-main"></a>
##### （3）设备管理与核函数调用

Host 准备与 CPU 相同的累计范围，申请设备内存并传入两个数值数组。`LaunchTwoStages` 先启动多个核局部求和，再在同一执行流启动一个核汇总；Host 等待两阶段完成后读取结果。

每次正式运行均核对两项总和，最后检查各核区间与局部结果。交易评价使用 NPU 返回的两项总和。核函数提交与等待计入 `avg_us`，数据准备、传输和检查在计时之外。`NpuResources` 集中释放设备资源，重点阅读 `LaunchTwoStages`、`RunNpu` 及核函数中的 TODO。

下面完整生成上述三部分代码。`TODO` 与 `END TODO` 标记待补全范围；保留注释标记，第四部分末尾的检查单元将读取其中的代码验证功能。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/ascend/stock_vwap_npu.asc
#include "kernel_operator.h"
#include <cstddef>
using namespace std;
using namespace AscendC;

// ==================== Device：NPU 核函数 ====================
extern "C" __global__ __vector__ void stock_partial_sum(
    GM_ADDR volume, GM_ADDR turnover, GM_ADDR partial, uint32_t n)
{
    // __vector__ 指定使用 Vector 核；这里用核内标量单元进行整数累加。
    InitSocState();
    // 本函数会在多个 Vector 核上同时执行，每个核取得自己的逻辑编号。
    uint32_t core = GetBlockIdx();
    uint32_t cores = GetBlockNum();

    // TODO 3A：计算每核至少处理的记录数和剩余记录数。
    uint32_t base = n / cores;
    uint32_t extra = n % cores;
    // END TODO 3A


    // 前 extra 个核各多处理一条记录，使各核记录数最多相差一条。
    // precedingExtra 表示排在当前核之前的额外记录数。
    uint32_t precedingExtra = extra;
    if (core < extra)
        precedingExtra = core;
    uint32_t begin = core * base + precedingExtra;
    uint32_t end = begin + base;
    if (core < extra)
        ++end;

    // GlobalTensor 将传入的 Device 地址包装为可按下标访问的数组。
    GlobalTensor<int64_t> volumeGm, turnoverGm, partialGm;
    volumeGm.SetGlobalBuffer((__gm__ int64_t *)volume, n);
    turnoverGm.SetGlobalBuffer((__gm__ int64_t *)turnover, n);
    partialGm.SetGlobalBuffer((__gm__ int64_t *)partial, cores * 4);
    // 输入由 Host 写入；标量读取前清理旧缓存，避免重复运行时读到旧数据。
    DataCacheCleanAndInvalid<int64_t, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(volumeGm);

    // 核之间并行；每个核只在自己的 [begin, end) 区间内顺序累加。
    int64_t volumeSum = 0;
    int64_t turnoverSum = 0;

    // TODO 4A：遍历当前核的区间，分别累加成交量和成交额。
    for (uint32_t i = begin; i < end; ++i) {
        volumeSum += volumeGm.GetValue(i);
        turnoverSum += turnoverGm.GetValue(i);
    }
    // END TODO 4A

    // 每核准备自己的 32 字节局部输出，避免各核写入同一位置。
    TPipe pipe;
    TBuf<QuePosition::VECOUT> outBuffer;
    pipe.InitBuffer(outBuffer, 32);
    LocalTensor<int64_t> outLocal = outBuffer.Get<int64_t>();
    outLocal.SetValue(0, volumeSum);
    outLocal.SetValue(1, turnoverSum);
    // 后两项记录区间边界，只用于检查，不计入最终两项总和。
    outLocal.SetValue(2, (int64_t)(begin));
    outLocal.SetValue(3, (int64_t)(end));
    // 等待 Scalar 写完局部缓冲区后，再由 MTE3 搬到全局内存。
    SetFlag<HardEvent::S_MTE3>(EVENT_ID0);
    WaitFlag<HardEvent::S_MTE3>(EVENT_ID0);
    // 偏移 core*4，保证各核写入互不重叠的 32 字节结果槽。
    DataCopy(partialGm[core * 4], outLocal, 4);
    PipeBarrier<PIPE_ALL>();
}

extern "C" __global__ __vector__ void stock_merge_sum(
    GM_ADDR partial, GM_ADDR result, uint32_t cores)
{
    // Host 仅启动一个 Vector 核，读取并合并第一阶段的结果。
    InitSocState();
    GlobalTensor<int64_t> partialGm, resultGm;
    partialGm.SetGlobalBuffer((__gm__ int64_t *)partial, cores * 4);
    resultGm.SetGlobalBuffer((__gm__ int64_t *)result, 4);

    // 预留最多 32 个核的局部结果空间，以及一组最终结果空间。
    TPipe pipe;
    TBuf<QuePosition::VECIN> inBuffer;
    TBuf<QuePosition::VECOUT> outBuffer;
    pipe.InitBuffer(inBuffer, 32 * 4 * sizeof(int64_t));
    pipe.InitBuffer(outBuffer, 4 * sizeof(int64_t));
    LocalTensor<int64_t> local = inBuffer.Get<int64_t>();
    LocalTensor<int64_t> output = outBuffer.Get<int64_t>();
    // 每核结果为 4 个 int64_t，整体搬入最多 32*32=1024 字节。
    DataCopy(local, partialGm, cores * 4);
    // MTE2 搬入完成后，Scalar 才能读取局部数据。
    SetFlag<HardEvent::MTE2_S>(EVENT_ID0);
    WaitFlag<HardEvent::MTE2_S>(EVENT_ID0);

    int64_t volumeSum = 0;
    int64_t turnoverSum = 0;
    // 每个结果槽只累加前两项；区间边界不参与求和。
    for (uint32_t core = 0; core < cores; ++core) {
        volumeSum += local.GetValue(core * 4);
        turnoverSum += local.GetValue(core * 4 + 1);
    }
    // 保留与第一阶段相同的 32 字节输出布局，前两项为有效总和。
    output.SetValue(0, volumeSum);
    output.SetValue(1, turnoverSum);
    output.SetValue(2, 0);
    output.SetValue(3, 0);
    // 标量写入完成后再搬出，确保 Host 取回的是本轮结果。
    SetFlag<HardEvent::S_MTE3>(EVENT_ID0);
    WaitFlag<HardEvent::S_MTE3>(EVENT_ID0);
    DataCopy(resultGm, output, 4);
    PipeBarrier<PIPE_ALL>();
}

// ==================== Host：CPU 上的设备管理与调用 ====================
// acl/acl.h 声明 CANN Runtime 接口；以下普通函数在 Host 上执行。
#include "trade_utils.hpp"
#include <acl/acl.h>
#include <iostream>
#include <sstream>
#include <stdexcept>
#include <exception>
using namespace std;

// 统一检查运行时接口的返回值，出错时报告具体操作。
static void CheckAcl(aclError code, const char *operation)
{
    if (code != ACL_SUCCESS) {
        ostringstream message;
        message << operation << " failed, ACL error=" << code;
        const char *detail = aclGetRecentErrMsg();
        if (detail != 0) message << "; " << detail;
        throw runtime_error(message.str());
    }
}
// ACL_CHECK 是本工程的宏：调用 Runtime 接口后统一检查返回值。
#define ACL_CHECK(operation) CheckAcl((operation), #operation)

// 辅助资源管理类：集中释放 Device 内存、执行流和设备上下文。
class NpuResources {
public:
    bool initialized;
    bool deviceSet;
    int device;
    aclrtStream stream;
    void *volume;
    void *turnover;
    void *partial;
    void *result;
    NpuResources() : initialized(false), deviceSet(false), device(0), stream(0),
                     volume(0), turnover(0), partial(0), result(0) {}
    ~NpuResources()
    {
        if (stream) (void)aclrtSynchronizeStream(stream);
        if (result) (void)aclrtFree(result);
        if (partial) (void)aclrtFree(partial);
        if (turnover) (void)aclrtFree(turnover);
        if (volume) (void)aclrtFree(volume);
        if (stream) (void)aclrtDestroyStream(stream);
        if (deviceSet) (void)aclrtResetDevice(device);
        if (initialized) (void)aclFinalize();
    }
private:
    NpuResources(const NpuResources &);
    NpuResources &operator=(const NpuResources &);
};

static void LaunchTwoStages(NpuResources &r, unsigned int n, unsigned int cores)
{
    // <<<计算块数, 保留参数, 执行流>>> 将核函数提交到指定执行流。
    // 多核读取不同区间，结果写入各核自己的 32 字节位置。
    stock_partial_sum<<<cores, 0, r.stream>>>(
        (GM_ADDR)r.volume, (GM_ADDR)r.turnover, (GM_ADDR)r.partial, n);
    // 同一执行流按顺序执行：第一阶段结束后才进行汇总。
    stock_merge_sum<<<1, 0, r.stream>>>(
        (GM_ADDR)r.partial, (GM_ADDR)r.result, cores);
    ACL_CHECK(aclrtSynchronizeStream(r.stream));
}

// CPU 按相同分工重算各核的局部结果，同时核对起止边界。
static bool CheckPartials(const NumericInput &input, const Options &opt,
                          const long long *partial)
{
    bool pass = true;
    for (int core = 0; core < opt.cores; ++core) {
        int begin, end;
        GetRange(input.length, opt.cores, core, begin, end);
        SumResult expected = {0, 0};
        for (int i = begin; i < end; ++i) {
            expected.volume += input.volume[i];
            expected.turnover += input.turnover[i];
        }
        int offset = core * PARTIAL_STRIDE;
        bool same = partial[offset] == expected.volume &&
                    partial[offset + 1] == expected.turnover &&
                    partial[offset + 2] == begin && partial[offset + 3] == end;
        if (!same) pass = false;
        if (opt.diagnostic) {
            cerr << "[CORE " << core << "] [" << partial[offset + 2] << ','
                 << partial[offset + 3] << ") volume=" << partial[offset]
                 << " turnover=" << partial[offset + 1] << " ";
            if (same)
                cerr << "PASS\n";
            else
                cerr << "FAIL\n";
        }
    }
    return pass;
}

static int RunNpu(const Options &opt, const SqList &prefix, const NumericInput &input)
{
    if (sizeof(long long) != 8)
        throw runtime_error("host long long must be 64-bit for this kernel ABI");
    SumResult reference = SumCpu(input);  // 仅作为正确性参考
    NpuResources r;
    ACL_CHECK(aclInit(0)); r.initialized = true;
    r.device = opt.device;
    // 这里使用程序可见的逻辑设备编号，当前教学环境为 0。
    uint32_t deviceCount = 0;
    ACL_CHECK(aclrtGetDeviceCount(&deviceCount));
    if (r.device < 0 || (uint32_t)(r.device) >= deviceCount)
        throw runtime_error("DEVICE_ID exceeds visible device count");
    ACL_CHECK(aclrtSetDevice(r.device)); r.deviceSet = true;
    ACL_CHECK(aclrtCreateStream(&r.stream));
    const char *soc = aclrtGetSocName();
    if (opt.diagnostic && soc) cerr << "[DEVICE] " << soc << '\n';

    // 申请 Device 存储，再把两列整数数组传到 NPU。
    size_t inputBytes = (size_t)(input.length) * sizeof(long long);
    size_t partialBytes = (size_t)(opt.cores) * PARTIAL_STRIDE * sizeof(long long);
    size_t resultBytes = PARTIAL_STRIDE * sizeof(long long);
    ACL_CHECK(aclrtMalloc(&r.volume, inputBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&r.turnover, inputBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&r.partial, partialBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&r.result, resultBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemcpy(r.volume, inputBytes, input.volume, inputBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(r.turnover, inputBytes, input.turnover, inputBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    // 先预热，再对两次核函数启动及等待完成的时间取平均。
    for (int i = 0; i < opt.warmup; ++i) {
        ACL_CHECK(aclrtMemset(r.partial, partialBytes, 0, partialBytes));
        ACL_CHECK(aclrtMemset(r.result, resultBytes, 0, resultBytes));
        LaunchTwoStages(r, input.length, opt.cores);
    }
    double totalUs = 0.0;
    bool allRoundsPass = true;
    long long output[PARTIAL_STRIDE] = {0, 0, 0, 0};
    for (int i = 0; i < opt.repeat; ++i) {
        // 每轮清理结果，避免把上轮的旧值误认为本轮计算结果；清理不计时。
        ACL_CHECK(aclrtMemset(r.partial, partialBytes, 0, partialBytes));
        ACL_CHECK(aclrtMemset(r.result, resultBytes, 0, resultBytes));
        double start = NowUs();
        LaunchTwoStages(r, input.length, opt.cores);
        totalUs += NowUs() - start;
        // 结果回传与校验放在计时区间之外。
        ACL_CHECK(aclrtMemcpy(output, resultBytes, r.result, resultBytes, ACL_MEMCPY_DEVICE_TO_HOST));
        if (output[0] != reference.volume || output[1] != reference.turnover)
            allRoundsPass = false;
    }
    double avgUs = totalUs / opt.repeat;
    long long partial[MAX_CORE_NUM * PARTIAL_STRIDE];
    ACL_CHECK(aclrtMemcpy(output, resultBytes, r.result, resultBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    ACL_CHECK(aclrtMemcpy(partial, partialBytes, r.partial, partialBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    // 业务计算使用返回值 output，而不是用 CPU 参考值替代 NPU 结果。
    SumResult result = {output[0], output[1]};
    bool pass = allRoundsPass && result.volume == reference.volume && result.turnover == reference.turnover;
    if (!CheckPartials(input, opt, partial)) pass = false;
    // 数据量少于核数时，部分核处理空区间，不计入实际有数据的核数。
    int activeCores = opt.cores;
    if (input.length < opt.cores)
        activeCores = input.length;
    string decision;
    if (pass)
        decision = EvaluateTrade(prefix.elem[prefix.length - 1], result);
    else
        decision = "未通过校验";
    if (result.volume > 0)
        WriteMetrics(opt.csvPath, "npu", opt, input.length, opt.cores, activeCores,
                     result, decision, avgUs, pass);
    if (!pass) {
        cerr << "[NPU] FAIL: totals or core partitions disagree with CPU reference\n";
        return 2;
    }
    if (opt.diagnostic)
        cerr << "[NPU] prefix=" << input.length << " cores=" << opt.cores
                  << " active_cores=" << activeCores << " avg_us=" << avgUs << " PASS\n";
    cout << decision << '\n';
    return 0;
}

// Host 主程序只组织数据与设备调用，实际求和在两阶段核函数中完成。
int main(int argc, char **argv)
{
    SqList all = {0, 0}, prefix = {0, 0};
    NumericInput input = {0, 0, 0};
    int exitCode = 0;
    try {
        Options opt = ParseArgs(argc, argv);
        LoadTrades(opt.dataPath, all);
        if (!BuildPrefix(all, opt.tradeId, prefix)) {
            cout << "查找失败\n";
        } else {
            ExtractNumeric(prefix, input);
            exitCode = RunNpu(opt, prefix, input);
        }
    } catch (const exception &error) {
        cerr << "[ERROR] " << error.what() << '\n';
        exitCode = 1;
    }
    DestroyNumeric(input); DestroyList_Sq(prefix); DestroyList_Sq(all);
    return exitCode;
}


#### 4.1.4 构建与运行脚本


<a id="seq-run-script"></a>
##### （1）运行脚本：`scripts/run.sh`

构建脚本先加载 CANN 环境，再查找毕昇编译器和原生 ASC CMake 软件包。`ASC_DIR` 指定 `ASCConfig.cmake` 所在目录，`CMAKE_ASC_ARCHITECTURES` 指定目标 NPU 架构，本实验的 Ascend 910B4 使用 `dav-2201`。脚本将命令行参数传给 NPU 主程序，在 `build_asc/` 中保存编译产物。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail
# 查找运行时、加载厂商环境，并为 CMake 指定原生 ASC 软件包。
is_cann_install_path() {
  local path="$1"
  [[ -f "${path}/include/acl/acl.h" || -f "${path}/runtime/include/acl/acl.h" ]]
}

detect_cann_install_path() {
  local arch base candidate
  arch=$(uname -m)
  if [[ -n "${1:-}" ]]; then
    for candidate in "${1}/${arch}-linux" "${1}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
    return 1
  fi
  for base in "${ASCEND_INSTALL_PATH:-}" "${ASCEND_HOME_PATH:-}" \
              "${ASCEND_CANN_PACKAGE_PATH:-}" "${ASCEND_TOOLKIT_HOME:-}" \
              /usr/local/Ascend/cann /usr/local/Ascend/cann/latest \
              /opt/conda/Ascend/cann-* /usr/local/Ascend/ascend-toolkit/latest \
              /usr/local/Ascend/ascend-toolkit/*; do
    [[ -n "${base}" ]] || continue
    for candidate in "${base}/${arch}-linux" "${base}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
  done
  return 1
}

prepare_cann_environment() {
  local path_hint="${1:-}" detected env_file="" candidate config=""
  if ! detected=$(detect_cann_install_path "${path_hint}"); then
    echo "Cannot find CANN. Set ASCEND_INSTALL_PATH to the installation directory." >&2
    return 1
  fi
  for candidate in "${detected}/set_env.sh" "${detected}/../set_env.sh"; do
    if [[ -f "${candidate}" ]]; then env_file="${candidate}"; break; fi
  done
  if [[ -n "${env_file}" ]]; then
    # 厂商脚本可能访问未定义变量，加载期间暂时关闭该检查。
    set +u
    source "${env_file}"
    set -u
  fi
  export ASCEND_INSTALL_PATH="${detected}"
  export ASCEND_CANN_PACKAGE_PATH="${detected}"

  # 同时支持常见的编译器安装布局；也允许使用 set_env.sh 配置的 PATH。
  for candidate in "${detected}/compiler/ccec_compiler/bin" \
                   "${detected}/ccec_compiler/bin" \
                   "${detected}/ascendc_devkit/ccec_compiler/bin"; do
    if [[ -x "${candidate}/bisheng" ]]; then
      export PATH="${candidate}:${PATH}"
      break
    fi
  done
  if ! command -v bisheng >/dev/null 2>&1; then
    echo "Cannot find bisheng. Install/load the CANN ASC development toolchain." >&2
    return 1
  fi

  # ASC_DIR 指向 ASCConfig.cmake 所在目录；未显式指定时在安装目录中查找。
  if [[ -n "${ASC_DIR:-}" ]]; then
    if [[ ! -f "${ASC_DIR}/ASCConfig.cmake" && ! -f "${ASC_DIR}/asc-config.cmake" ]]; then
      echo "ASC_DIR does not contain an ASC CMake package: ${ASC_DIR}" >&2
      return 1
    fi
  else
    config=$(find -L "${detected}" -type f \
      \( -name ASCConfig.cmake -o -name asc-config.cmake \) -print -quit 2>/dev/null || true)
    if [[ -z "${config}" ]]; then
      echo "Cannot find the native ASC CMake package. Install it or set ASC_DIR." >&2
      return 1
    fi
    ASC_DIR=$(dirname "${config}")
  fi
  export ASC_DIR
  export CMAKE_PREFIX_PATH="${detected}:${CMAKE_PREFIX_PATH:-}"
  export LD_LIBRARY_PATH="${detected}/lib64:${detected}/runtime/lib64:${LD_LIBRARY_PATH:-}"
  # Ascend 910B4 对应 dav-2201；其他架构可通过环境变量显式指定。
  export CMAKE_ASC_ARCHITECTURES="${CMAKE_ASC_ARCHITECTURES:-dav-2201}"
  echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"
  echo "[INFO] ASC_DIR=${ASC_DIR}"
  echo "[INFO] CMAKE_ASC_ARCHITECTURES=${CMAKE_ASC_ARCHITECTURES}"
}

# 构建目录单独保存 ASC 工程的配置和编译产物。
run_experiment_main() {
  local ROOT BUILD_DIR
  ROOT=$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)
  BUILD_DIR="${ROOT}/build_asc"
  prepare_cann_environment "${ASCEND_INSTALL_PATH:-}"
  cmake -S "${ROOT}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release \
    -DBUILD_ASCEND=ON -DCMAKE_ASC_RUN_MODE=npu \
    -DCMAKE_ASC_ARCHITECTURES="${CMAKE_ASC_ARCHITECTURES}" -DASC_DIR="${ASC_DIR}"
  cmake --build "${BUILD_DIR}" -j 4
  mkdir -p "${ROOT}/results"
  "${BUILD_DIR}/stock_vwap_npu" "$@"
}
if [[ "${BASH_SOURCE[0]}" == "$0" ]]; then run_experiment_main "$@"; fi


### 4.2 项目构建

先生成项目根目录下的 `CMakeLists.txt`，检查工程文件，再编译并运行 CPU 基本测试。NPU 构建在第 4.3.2 节由脚本执行。


<a id="seq-cmake"></a>
#### 4.2.1 CMake 配置文件：`CMakeLists.txt`

项目名称与第 1 节保持一致，为 `stock_trading_evaluation_seq`。本文件分别配置 CPU 公共库、运行程序、测试程序及 NPU 核函数库的编译与链接；涉及 Ascend C SDK 的目标采用工具链要求的编译设置。

`BUILD_ASCEND=OFF` 时无需 CANN，可独立构建 CPU 部分；开启后编译两个核函数库，生成对应的启动接口，并链接 NPU 运行程序。

本文件作为辅助代码完整提供，无需补全。


CMake 用于组织源文件和编译依赖。`BUILD_ASCEND=OFF` 时仅使用 C++ 编译器构建公共库、CPU 程序和检查程序；开启该选项后，先通过 `find_package(ASC REQUIRED)` 加载工具链，再启用 ASC 语言并编译 `.asc` 文件。NPU 程序链接同一份公共算法库；CPU 目标与 ASC 目标分别采用各自的编译配置。


In [ ]:
%%writefile Source/stock_trading_evaluation_seq/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
option(BUILD_ASCEND "Build Ascend C NPU program" OFF)

# 只有 NPU 构建才加载 ASC 工具链，CPU 构建不依赖 CANN。
if(BUILD_ASCEND)
  set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture")
  set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "ASC run mode")
  find_package(ASC REQUIRED)
  project(stock_trading_evaluation_seq LANGUAGES ASC CXX)
else()
  project(stock_trading_evaluation_seq LANGUAGES CXX)
endif()

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

# 公共库与 CPU 程序可在没有 CANN 的环境中独立构建。
add_library(stock_common STATIC src/seq_list.cpp src/trade_utils.cpp)
target_include_directories(stock_common PUBLIC include)
set_target_properties(stock_common PROPERTIES CXX_STANDARD 98 CXX_STANDARD_REQUIRED YES CXX_EXTENSIONS YES)
target_compile_options(stock_common PRIVATE -O2 -Wall -Wextra)
add_executable(stock_vwap_cpu src/cpu_main.cpp)
add_executable(stock_common_test src/test_common.cpp)
foreach(target stock_vwap_cpu stock_common_test)
  target_link_libraries(${target} PRIVATE stock_common)
  set_target_properties(${target} PROPERTIES CXX_STANDARD 98 CXX_STANDARD_REQUIRED YES CXX_EXTENSIONS YES)
endforeach()
if(UNIX AND NOT APPLE)
  target_link_libraries(stock_common PUBLIC rt)
endif()

if(BUILD_ASCEND)
  # .asc 由毕昇编译器同时处理 Host 与 Device；通过 ASC 驱动完成链接。
  add_executable(stock_vwap_npu ascend/stock_vwap_npu.asc)
  set_target_properties(stock_vwap_npu PROPERTIES LINKER_LANGUAGE ASC)
  # 将目标 NPU 架构显式传给 ASC 编译器。
  target_compile_options(stock_vwap_npu PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>)
  target_include_directories(stock_vwap_npu PRIVATE include)
  target_link_libraries(stock_vwap_npu PRIVATE stock_common)
endif()


<a id="seq-build-check"></a>
#### 4.2.2 工程检查与 CPU 编译

检查所有工程文件是否已生成，再使用 CMake 编译 CPU 程序和测试程序。若本地未安装 CMake，下面的单元格直接调用 C++ 编译器完成相同的 CPU 编译检查。NPU 工程仍需要 CMake 和 CANN。

本步骤检查顺序表和 CPU 算法，不代表 NPU 已执行成功。



In [ ]:
# 检查项目文件与脚本，再构建 CPU 程序并运行基本测试。
import csv
required = [
    'include/trades.hpp', 'include/seq_list.hpp', 'include/trade_utils.hpp',
    'src/seq_list.cpp', 'src/trade_utils.cpp', 'src/cpu_main.cpp', 'src/test_common.cpp',
    'ascend/stock_vwap_npu.asc', 'scripts/run.sh', 'CMakeLists.txt'
]
missing = [name for name in required if not (WORK_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('尚未生成的工程文件：' + ', '.join(missing))
subprocess.run(['bash', '-n', str(WORK_DIR / 'scripts/run.sh')], check=True)

CPU_BUILD = WORK_DIR / 'build_cpu'
CPU_BUILD.mkdir(exist_ok=True)
CPU_BIN = CPU_BUILD / 'stock_vwap_cpu'
TEST_BIN = CPU_BUILD / 'stock_common_test'
cmake_command = shutil.which('cmake')
if cmake_command:
    subprocess.run([cmake_command, '-S', str(WORK_DIR), '-B', str(CPU_BUILD),
                    '-DBUILD_ASCEND=OFF', '-DCMAKE_BUILD_TYPE=Release',
                    '-DCMAKE_EXPORT_COMPILE_COMMANDS=ON'], check=True)
    subprocess.run([cmake_command, '--build', str(CPU_BUILD), '-j', '4'], check=True)
else:
    cxx = shutil.which('g++') or shutil.which('c++')
    if not cxx:
        raise RuntimeError('未找到 g++ 或 c++ 编译器。')
    common_args = [cxx, '-std=gnu++98', '-O2', '-Wall', '-Wextra',
                   '-I', str(WORK_DIR / 'include'),
                   str(WORK_DIR / 'src/seq_list.cpp'), str(WORK_DIR / 'src/trade_utils.cpp')]
    for name, executable in [('cpu_main.cpp', CPU_BIN), ('test_common.cpp', TEST_BIN)]:
        command = common_args + [str(WORK_DIR / 'src' / name), '-o', str(executable)]
        if platform.system() == 'Linux':
            command += ['-lrt']
        subprocess.run(command, check=True)
subprocess.run([str(TEST_BIN)], check=True)
print('工程文件、脚本语法与 CPU 基本测试检查通过。')


### 4.3 运行并输出实验结果

本节使用第 2.3 节的交易编号、核数和计时参数，先运行 CPU，再运行 NPU。两条路径使用同一文件、同一交易编号和同一累计范围。

<a id="seq-run-cpu"></a>

#### 4.3.1 CPU 参考运行

标准输出只有交易评估标签，累计值与执行时间保存到 `results/cpu_result.csv`，供第 4 部分读取。

In [ ]:
# 调用 CPU 程序，标准输出为评价标签，实验记录另存为 CSV。
CPU_CSV = WORK_DIR / 'results/cpu_result.csv'
# 删除本次运行的旧结果，避免把以前的结果当成本次输出。
if CPU_CSV.exists():
    CPU_CSV.unlink()
cpu_command = [str(CPU_BIN), '--data', str(DATA_FILE), '--trade-id', str(TRADE_ID),
               '--warmup', str(WARMUP), '--repeat', str(REPEAT), '--csv', str(CPU_CSV)]
completed = subprocess.run(cpu_command, check=True, capture_output=True, text=True)
print(completed.stdout.strip())


<a id="seq-run-npu"></a>
#### 4.3.2 NPU 构建与运行

脚本依次完成 CANN 环境准备、CMake 配置、核函数编译及设备运行。程序启动 32 核局部求和，再启动单核汇总，取回总和后完成校验和交易评估。实际核数以 `CORE_NUM` 为准。

当前教学环境中，程序使用的逻辑设备编号为 0，因此在本运行单元格中设置 `DEVICE_ID = 0`。它与 `npu-smi` 显示的物理卡号分别使用不同的编号体系。

辅助诊断会输出各核区间、局部结果及校验状态；正常评估标签仍单独输出。结果保存到 `results/npu_result.csv`。设备初始化、构建或核函数校验失败时会明确报错；缺少 NPU 环境时明确提示跳过。



In [ ]:
# 在有 CANN 和设备工具时构建并实际运行 NPU 程序。
DEVICE_ID = 0  # 程序可见的逻辑设备编号
NPU_BIN = WORK_DIR / 'build_asc/stock_vwap_npu'
NPU_CSV = WORK_DIR / 'results/npu_result.csv'
NPU_RUN_OK = False
if NPU_CSV.exists():
    NPU_CSV.unlink()
if HAS_ASCEND:
    run_env = os.environ.copy()
    if SOC_VERSION:
        run_env['SOC_VERSION'] = SOC_VERSION
    npu_command = ['bash', str(WORK_DIR / 'scripts/run.sh'), '--data', str(DATA_FILE),
                   '--trade-id', str(TRADE_ID), '--cores', str(CORE_NUM),
                   '--device', str(DEVICE_ID), '--warmup', str(WARMUP),
                   '--repeat', str(REPEAT), '--csv', str(NPU_CSV), '--diagnostic']
    subprocess.run(npu_command, env=run_env, check=True)
    NPU_RUN_OK = True
else:
    print('跳过 NPU 编译与运行：当前环境未检测到 CANN 与 NPU 工具。')


### 4.4 实操任务检查

下面按任务一至任务五的顺序集中检查 TODO。请先生成对应项目文件，再依次执行这些检查单元。检查只在临时目录编译和运行测试，不修改项目文件，也不控制前面的构建与运行。任务三、四在 CPU 上验证填写语句的计算功能，实际 NPU 执行由 4.3 节检查。


<a id="task1-check"></a>
**任务一检查：**

运行下面的检查单元，核对任务一补全的代码是否正确。通过时输出 `Task 1 PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。

In [ ]:
# 本单元只检查 TODO；临时测试文件与可执行程序不会写入项目目录。
def _seq_check_todo(label, source, project_sources=(), required_files=()):
    from pathlib import Path
    import shutil
    import subprocess
    import tempfile

    try:
        project = Path('Source/stock_trading_evaluation_seq').resolve()
        missing = [name for name in required_files if not (project / name).is_file()]
        if missing:
            print(label + ' SKIPPED：请先运行对应的文件生成单元。缺少：' + '、'.join(missing))
            return 'SKIPPED'
        compiler = shutil.which('g++') or shutil.which('c++')
        if compiler is None:
            print(label + ' SKIPPED：未找到 C++ 编译器，尚不能判断 TODO 是否正确。')
            return 'SKIPPED'

        # 每次重新编译当前代码，不使用先前构建的程序或先前的检查结果。
        with tempfile.TemporaryDirectory(prefix='stock_seq_todo_') as directory:
            directory = Path(directory)
            test_file = directory / 'todo_check.cpp'
            executable = directory / 'todo_check'
            test_file.write_text(source, encoding='utf-8')
            command = [compiler, '-std=gnu++98', '-O0', '-Wall', '-Wextra',
                       '-I', str(project / 'include'), str(test_file)]
            command += [str(project / name) for name in project_sources]
            command += ['-o', str(executable)]
            built = subprocess.run(command, cwd=str(directory), capture_output=True,
                                   text=True, errors='replace', timeout=30, check=False)
            if built.returncode != 0:
                print(label + ' FAILED：检查程序编译未通过，请查看成员定义或函数中的代码。')
                print((built.stdout + built.stderr)[-6000:])
                return 'FAILED'
            run = subprocess.run([str(executable)], cwd=str(directory), capture_output=True,
                                 text=True, errors='replace', timeout=10, check=False)
            completed = 'SEQ_TODO_CHECK_PASSED' in run.stdout.splitlines()
            if run.returncode == 0 and completed:
                print(label + ' PASSED')
                return 'PASSED'
            print(label + ' FAILED：功能检查未通过。')
            detail = (run.stdout + run.stderr).strip()
            print(detail[-6000:] if detail else '检查程序未正常完成，返回码：' + str(run.returncode))
            return 'FAILED'
    except subprocess.TimeoutExpired:
        print(label + ' FAILED：检查超时，请检查循环条件或运行环境。')
        return 'FAILED'
    except Exception as error:
        print(label + ' SKIPPED：检查未能完成，原因：' + str(error))
        return 'SKIPPED'

_seq_task1_source = r'''
#include "trades.hpp"
#include <iostream>
using namespace std;

// 引用参数用于核对 elem 与 length 的准确类型，此处无需学生补全。
static void SetMembers(trades *&elem, int &length, trades records[])
{
    elem = records;
    length = 2;
}

int main()
{
    trades records[2];
    records[0].trade_id = 101;
    records[1].trade_id = 102;
    SqList L = {0, 0};
    SetMembers(L.elem, L.length, records);
    if (L.elem != records || L.length != 2 ||
        L.elem[0].trade_id != 101 || L.elem[1].trade_id != 102) {
        cerr << "顺序表应通过 elem 访问成交记录，并用 length 保存当前表长。\n";
        return 1;
    }
    cout << "SEQ_TODO_CHECK_PASSED\n";
    return 0;
}
'''

_seq_task1_status = _seq_check_todo(
    'Task 1', _seq_task1_source, required_files=('include/trades.hpp',))


# 任务三、四读取实际核函数中的 TODO，只在 CPU 上检查语句功能。
# 测试输入与判定条件由检查程序独立给出，不比较代码是否与答案相同。
def _seq_check_kernel_todo(label, todo, driver_source):
    from pathlib import Path
    import re

    if '_seq_check_todo' not in globals():
        print(label + ' SKIPPED：请先运行任务一检查单元，加载辅助检查函数。')
        return 'SKIPPED'
    try:
        kernel = Path('Source/stock_trading_evaluation_seq/ascend/stock_vwap_npu.asc')
        if not kernel.is_file():
            print(label + ' SKIPPED：请先生成 stock_vwap_npu.asc。')
            return 'SKIPPED'
        source = kernel.read_text(encoding='utf-8')
        pattern = (r'^[ \t]*// TODO ' + re.escape(todo) + r'[：:][^\n]*\n'
                   r'(.*?)^[ \t]*// END TODO ' + re.escape(todo) + r'[ \t]*$')
        match = re.search(pattern, source, flags=re.M | re.S)
        if match is None:
            print(label + ' FAILED：未找到 TODO ' + todo + ' 的完整注释标记。')
            return 'FAILED'
        driver = driver_source.replace('// __STUDENT_TODO__', match.group(1))
        return _seq_check_todo(label, driver)
    except Exception as error:
        print(label + ' SKIPPED：检查未能完成，原因：' + str(error))
        return 'SKIPPED'


<a id="task2-check"></a>

**任务二检查:**

运行下面的检查单元，核对任务二补全的代码是否正确。通过时输出 `Task 2 PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。


In [ ]:
# 分别编译当前顺序表实现，独立检查初始化与构建操作。
_seq_task2a_source = r'''
#include "seq_list.hpp"
#include <iostream>
#include <stdexcept>
using namespace std;

static void Check(bool condition, const char *message)
{
    if (!condition) throw runtime_error(message);
}

int main()
{
    // 先给表长一个非零值，检查初始化是否确实将其置为 0。
    SqList L = {0, 7};
    int result = 0;
    try {
        Check(InitList_Sq(L), "初始化应成功并返回 true。");
        Check(L.elem != 0, "尚未为 elem 分配成交记录数组。");
        Check(L.length == 0, "空表的 length 应为 0。");
        // 检查新申请的存储空间能够保存成交记录。
        L.elem[0].trade_id = 101;
        Check(L.elem[0].trade_id == 101, "新申请的存储空间应能保存成交记录。");
        Check(L.length == 0, "访问存储空间不应改变有效记录数。");
        cout << "SEQ_TODO_CHECK_PASSED\n";
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        result = 1;
    }
    DestroyList_Sq(L);
    return result;
}
'''

_seq_task2b_source = r'''
#include "seq_list.hpp"
#include <iostream>
#include <stdexcept>
using namespace std;

static void Check(bool condition, const char *message)
{
    if (!condition) throw runtime_error(message);
}

static trades MakeTrade(int id)
{
    trades record;
    record.trade_id = id;
    if (id % 2 == 0)
        record.stockcode = "AAPL";
    else
        record.stockcode = "MSFT";
    record.date = "20191230";
    record.time = 34200000000000LL + id;
    record.volume_t = id + 1;
    record.price = 1000000 + id;
    record.turnover_t = (long long)(record.volume_t) * record.price;
    return record;
}

static bool SameTrade(const trades &left, const trades &right)
{
    return left.trade_id == right.trade_id && left.stockcode == right.stockcode &&
           left.date == right.date && left.time == right.time &&
           left.volume_t == right.volume_t && left.price == right.price &&
           left.turnover_t == right.turnover_t;
}

int main()
{
    // 检查程序直接准备空表，使构建检查不依赖 TODO 2A 的运行结果。
    SqList L = {new trades[MAXSIZE], 0};
    int result = 0;
    try {
        trades records[4];
        const int ids[4] = {103, 101, 104, 102};
        for (int i = 0; i < 4; ++i) records[i] = MakeTrade(ids[i]);

        Check(CreateList_Sq(L, 0, 0) && L.length == 0,
              "没有输入记录时，顺序表应保持为空。");
        Check(CreateList_Sq(L, records, 1) && L.length == 1,
              "存入一条记录后，表长应为 1。");
        Check(SameTrade(L.elem[0], records[0]), "未完整保存第一条成交记录。");

        DestroyList_Sq(L);
        L.elem = new trades[MAXSIZE];
        L.length = 0;
        // 用已知记录填充测试位置，避免错误实现读取未赋值的数据。
        for (int i = 0; i < 4; ++i) L.elem[i] = MakeTrade(900 + i);
        Check(CreateList_Sq(L, records, 4) && L.length == 4,
              "应依次存入全部记录，并使表长等于记录数。");
        for (int i = 0; i < 4; ++i) {
            Check(SameTrade(L.elem[i], MakeTrade(ids[i])),
                  "构建后记录的内容或先后顺序不正确。");
            Check(SameTrade(records[i], MakeTrade(ids[i])),
                  "构建过程不应修改输入记录数组。");
        }
        cout << "SEQ_TODO_CHECK_PASSED\n";
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        result = 1;
    }
    DestroyList_Sq(L);
    return result;
}
'''

# 两项独立检查；不会把状态传给后续的工程构建和运行单元。
if '_seq_check_todo' in globals():
    _seq_task2a_status = _seq_check_todo(
        'TODO 2A', _seq_task2a_source, project_sources=('src/seq_list.cpp',),
        required_files=('include/trades.hpp', 'include/seq_list.hpp', 'src/seq_list.cpp'))
    _seq_task2b_status = _seq_check_todo(
        'TODO 2B', _seq_task2b_source, project_sources=('src/seq_list.cpp',),
        required_files=('include/trades.hpp', 'include/seq_list.hpp', 'src/seq_list.cpp'))
    if _seq_task2a_status == 'PASSED' and _seq_task2b_status == 'PASSED':
        print('Task 2 PASSED')
    elif 'FAILED' in (_seq_task2a_status, _seq_task2b_status):
        print('Task 2 FAILED：请根据上方提示检查 TODO 2A、TODO 2B 或其依赖的定义。')
    else:
        print('Task 2 SKIPPED：尚未完成全部检查，请补齐检查环境后重试。')
else:
    print('Task 2 SKIPPED：请先运行任务一检查单元，加载辅助检查函数。')


<a id="task3-check"></a>

**任务三检查：**

运行下面的检查单元，核对 `TODO 3A` 中的记录数分配是否正确。使用记录数为 0、少于核数、能够整除及不能整除等输入，检查基本记录数、剩余记录数以及分配后的完整覆盖情况。

本单元读取当前生成的 `stock_vwap_npu.asc` 中的 `stock_partial_sum` 中的 `TODO 3A`，在 CPU 上检查其运算逻辑。通过时输出 `Task 3 PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。检查状态不会影响后续文件生成、项目构建和运行。

**注：** 此处通过仅表示分块语句的功能正确；真实 NPU 上的分工与运行结果仍需在第 4.3.2 节和第 5.2 节验证。


In [ ]:
# 检查代码只用于验证 TODO，不改动项目文件和构建配置。
_seq_task3_source = r'''
#include <stdint.h>
#include <iostream>
#include <stdexcept>
using namespace std;

// 检查程序只负责提供参数，待检语句取自学生生成的核函数文件。
static void StudentSplit(uint32_t n, uint32_t cores,
                         uint32_t &outBase, uint32_t &outExtra)
{
    // __STUDENT_TODO__
    outBase = base;
    outExtra = extra;
}

static void Check(bool condition, const char *message)
{
    if (!condition) throw runtime_error(message);
}

int main()
{
    // 覆盖空数据、记录数少于核数、整除和有余数等情况。
    const uint32_t sizes[] = {0, 1, 2, 17, 31, 32, 33, 1000, 67077, 200000};
    try {
        for (int i = 0; i < 10; ++i) {
            uint32_t n = sizes[i];
            for (uint32_t cores = 1; cores <= 32; ++cores) {
                uint32_t base = 0, extra = 0;
                StudentSplit(n, cores, base, extra);
                Check(extra < cores, "剩余记录数必须小于计算核数量。");
                // 用数量关系检查结果，而不是比较学生代码的文字。
                unsigned long long covered = (unsigned long long)base * cores + extra;
                Check(covered == n, "基本记录数和剩余记录数不能完整覆盖输入。");
                uint32_t begin = 0;
                for (uint32_t core = 0; core < cores; ++core) {
                    uint32_t count = base;
                    if (core < extra) ++count;
                    uint32_t end = begin + count;
                    Check(begin <= end && end <= n, "分块范围超出记录总数。");
                    begin = end;
                }
                Check(begin == n, "分块后存在遗漏或重复的记录。");
            }
        }
        cout << "SEQ_TODO_CHECK_PASSED\n";
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        return 1;
    }
    return 0;
}
'''

if '_seq_check_kernel_todo' in globals():
    _seq_task3_status = _seq_check_kernel_todo(
        'Task 3', '3A', _seq_task3_source)
else:
    print('Task 3 SKIPPED：请先运行任务一检查单元，加载辅助检查函数。')


<a id="task4-check"></a>

**任务四检查：**

运行下面的检查单元，核对 `TODO 4A` 是否正确累加 `[begin, end)` 内的成交量和成交额。检查包含空区间、单条记录、多个不同区间以及超过 32 位整数范围的累计值。

本单元读取当前生成的 `stock_vwap_npu.asc` 中的 `stock_partial_sum` 中的 `TODO 4A`，用 CPU 数组提供 `GetValue` 读取方式，再与独立的区间合计比较。通过时输出 `Task 4 PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。检查状态不会影响后续文件生成、项目构建和运行。

**注：** 本单元只检查循环的计算逻辑，不执行 Ascend C，也不验证 NPU 的数据搬运、同步或并行性能；这些内容由后续 NPU 运行检查。


In [ ]:
# 检查代码只用于验证 TODO，不改动项目文件和构建配置。
_seq_task4_source = r'''
#include <stdint.h>
#include <iostream>
#include <stdexcept>
using namespace std;

// CPU 检查只模拟 GetValue 的读取含义，不模拟 NPU 的并行执行和性能。
class CheckArray {
private:
    const int64_t *values;
    uint32_t length;
public:
    CheckArray(const int64_t *data, uint32_t n) : values(data), length(n) {}
    int64_t GetValue(uint32_t i) const
    {
        if (i >= length) throw runtime_error("求和循环访问了数组范围之外的位置。");
        return values[i];
    }
};

struct CheckSum {
    int64_t volume;
    int64_t turnover;
};

static CheckSum StudentLocalSum(const CheckArray &volumeGm, const CheckArray &turnoverGm,
                               uint32_t begin, uint32_t end)
{
    int64_t volumeSum = 0;
    int64_t turnoverSum = 0;
    // __STUDENT_TODO__
    CheckSum result = {volumeSum, turnoverSum};
    return result;
}

int main()
{
    const uint32_t n = 257;
    int64_t volume[n], turnover[n];
    int64_t volumePrefix[n + 1], turnoverPrefix[n + 1];
    volumePrefix[0] = 0;
    turnoverPrefix[0] = 0;
    // 两列使用不同数值，并让累计值超过 int 范围。
    for (uint32_t i = 0; i < n; ++i) {
        volume[i] = 1000000000LL + i;
        turnover[i] = volume[i] * (1000000LL + i);
        volumePrefix[i + 1] = volumePrefix[i] + volume[i];
        turnoverPrefix[i + 1] = turnoverPrefix[i] + turnover[i];
    }
    CheckArray volumeGm(volume, n), turnoverGm(turnover, n);
    const uint32_t spans[] = {0, 1, 2, 7, 31, 32, 33, 257};
    try {
        // 检查空区间、单元素区间、中间区间及完整区间。
        for (uint32_t begin = 0; begin <= n; ++begin) {
            for (int j = 0; j < 8; ++j) {
                uint32_t end = begin + spans[j];
                if (end > n) end = n;
                CheckSum result = StudentLocalSum(volumeGm, turnoverGm, begin, end);
                // 用独立构建的累计数组计算参照值。
                if (result.volume != volumePrefix[end] - volumePrefix[begin] ||
                    result.turnover != turnoverPrefix[end] - turnoverPrefix[begin])
                    throw runtime_error("局部合计不正确，请检查区间边界和两项累加语句。");
            }
        }
        cout << "SEQ_TODO_CHECK_PASSED\n";
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        return 1;
    }
    return 0;
}
'''

if '_seq_check_kernel_todo' in globals():
    _seq_task4_status = _seq_check_kernel_todo(
        'Task 4', '4A', _seq_task4_source)
else:
    print('Task 4 SKIPPED：请先运行任务一检查单元，加载辅助检查函数。')


<a id="task5-check"></a>

**任务五检查：**

运行下面的检查单元，核对 `TODO 5A` 中的 VWAP 计算与卖方评价是否正确。检查成交价高于、等于和低于 VWAP 的情况，同时包含带有小数的 VWAP 和超过 `int` 范围的累计成交量。

本单元编译当前项目中的 `EvaluateTrade` 函数，直接检查其返回结果。通过时输出 `Task 5 PASSED`，未通过时显示 `FAILED`；缺少文件或编译环境时显示 `SKIPPED`。检查状态不会影响后续文件生成、项目构建和运行。


In [ ]:
# 检查代码只用于验证 TODO，不改动项目文件和构建配置。
_seq_task5_source = r'''
#include "trade_utils.hpp"
#include <iostream>
#include <stdexcept>
using namespace std;

int main()
{
    // 同时检查正常累计量、非整数 VWAP，以及超过 int 范围的累计成交量。
    const SumResult cases[] = {
        {100, 100000000LL},
        {3, 3000001LL},
        {3000000000LL, 3000000000000000LL}
    };
    const int prices[] = {999999, 1000000, 1000001};
    try {
        trades target;
        target.trade_id = 1;
        target.stockcode = "TEST";
        target.date = "20191230";
        target.time = 34200000000000LL;
        target.volume_t = 1;
        for (int i = 0; i < 3; ++i) {
            for (int j = 0; j < 3; ++j) {
                target.price = prices[j];
                target.turnover_t = target.price;
                // 这些人工数据的乘积均在 long long 范围内。
                // 用整数关系形成独立参照，检查实际 EvaluateTrade 函数。
                string expected;
                if ((long long)target.price * cases[i].volume > cases[i].turnover)
                    expected = "投资成功";
                else
                    expected = "投资失败";
                if (EvaluateTrade(target, cases[i]) != expected)
                    throw runtime_error("评估结果不正确，请检查 VWAP 计算、比较方向及相等时的规则。");
            }
        }
        cout << "SEQ_TODO_CHECK_PASSED\n";
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        return 1;
    }
    return 0;
}
'''

if '_seq_check_todo' in globals():
    _seq_task5_status = _seq_check_todo(
        'Task 5', _seq_task5_source,
        project_sources=('src/seq_list.cpp', 'src/trade_utils.cpp'),
        required_files=('include/trades.hpp', 'include/seq_list.hpp',
                        'include/trade_utils.hpp', 'src/seq_list.cpp', 'src/trade_utils.cpp'))
else:
    print('Task 5 SKIPPED：请先运行任务一检查单元，加载辅助检查函数。')


---
## 5. 性能对比分析

本节使用默认交易编号 `199016356`，分别运行 CPU 与 NPU 版本，展示交易评价结果，并比较求和计算的平均耗时。

### 5.1 设置本次运行参数

先完成第 4 部分的项目文件生成和 CPU 工程构建，再依次运行第 5.1～5.4 节。交易编号已在下面代码中给定，无需交互输入；核数、预热次数和正式计时次数沿用第 2.3 节设置。

本节的 `PERF_TRADE_ID` 专用于本次性能对比。每次运行第 5.1 节都会清空本轮结果记录，随后依次执行 CPU、NPU 和结果展示单元，确保表格与图形使用本次实际运行的数据。


In [ ]:
# 使用给定的默认交易编号，无需学生交互输入。
from pathlib import Path
import csv
import math
import os
import subprocess

PERF_TRADE_ID = 199016356  # 本次性能对比使用的默认交易编号

PERF_CORES = int(CORE_NUM)
PERF_WARMUP = int(WARMUP)
PERF_REPEAT = int(REPEAT)
if not 1 <= PERF_CORES <= 32 or PERF_WARMUP < 0 or PERF_REPEAT < 1:
    raise ValueError('请检查第 2.3 节的核数、预热次数与正式计时次数。')

PERF_PROJECT = Path(WORK_DIR).resolve()
PERF_DATA = Path(DATA_FILE).resolve()
PERF_DIR = PERF_PROJECT / 'results'
PERF_DIR.mkdir(parents=True, exist_ok=True)
PERF_CPU_CSV = PERF_DIR / ('performance_%d_cpu.csv' % PERF_TRADE_ID)
PERF_NPU_CSV = PERF_DIR / ('performance_%d_npu.csv' % PERF_TRADE_ID)
PERF_RESULTS = {}  # 只保存本轮实际运行的结果。
PERF_CONFIG = (PERF_TRADE_ID, PERF_CORES, PERF_WARMUP, PERF_REPEAT, str(PERF_DATA))

# 以下辅助函数负责运行程序和读取计时结果，学生无需补全。
def _perf_arguments():
    return ['--data', str(PERF_DATA), '--trade-id', str(PERF_TRADE_ID),
            '--warmup', str(PERF_WARMUP), '--repeat', str(PERF_REPEAT)]


def _perf_execute(name, command, output_file, env=None):
    # 先清除本轮同名文件，防止误读上次运行留下的结果。
    PERF_RESULTS.pop(name, None)
    if output_file.exists():
        output_file.unlink()
    print('正在运行 %s 版本……' % name, flush=True)
    completed = subprocess.run(command, env=env, capture_output=True,
                               text=True, errors='replace', check=False)
    if completed.returncode:
        PERF_RESULTS[name] = {'state': 'FAILED', 'message': '程序运行失败'}
        print((completed.stdout + '\n' + completed.stderr).strip()[-8000:])
        raise RuntimeError('%s 运行失败，请根据上方实际日志排查。' % name)
    if not output_file.is_file():
        if '查找失败' in completed.stdout.splitlines():
            PERF_RESULTS[name] = {'state': 'NOT_FOUND', 'message': '查找失败'}
            print('%s 执行结果：查找失败；本次没有可用于性能比较的记录。' % name)
            return
        PERF_RESULTS[name] = {'state': 'FAILED', 'message': '没有生成计时结果'}
        raise RuntimeError('%s 未生成计时文件：%s' % (name, output_file))

    with output_file.open(encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != 1:
        raise RuntimeError('%s 计时文件应只有一条本次运行结果。' % name)
    row = rows[0]
    required = ['implementation', 'trade_id', 'n', 'cores', 'active_cores',
                'volume_sum', 'turnover_sum', 'vwap', 'decision', 'avg_us', 'status']
    if any(field not in row for field in required):
        raise RuntimeError('%s 计时文件字段不完整。' % name)
    for field in ['trade_id', 'n', 'cores', 'active_cores',
                  'volume_sum', 'turnover_sum']:
        row[field] = int(row[field])
    row['avg_us'] = float(row['avg_us'])
    row['vwap'] = float(row['vwap'])
    expected_implementation = 'cpu' if name == 'CPU' else 'npu'
    expected_cores = 1 if name == 'CPU' else PERF_CORES
    if (row['trade_id'] != PERF_TRADE_ID or row['implementation'] != expected_implementation or
        row['n'] < 1 or row['cores'] != expected_cores or
        row['active_cores'] != min(row['n'], expected_cores) or
        row['status'] != 'PASS' or row['decision'] not in ['投资成功', '投资失败'] or
        not math.isfinite(row['avg_us']) or row['avg_us'] < 0 or
        not math.isfinite(row['vwap']) or row['volume_sum'] <= 0):
        raise RuntimeError('%s 的结果或计时记录无效，请查看实际程序输出。' % name)
    # 数值来自当前 C++ 程序的 CSV；不使用 Notebook 的墙钟时间代替算法计时。
    PERF_RESULTS[name] = {'state': 'OK', 'row': row, 'config': PERF_CONFIG}
    print('%s 执行结果：%s；VWAP = %.8f 美元/股；平均耗时 = %.3f μs。' %
          (name, row['decision'], row['vwap'], row['avg_us']))


print('本次交易编号：%d；NPU 最多 %d 核；预热 %d 次，正式计时 %d 次。' %
      (PERF_TRADE_ID, PERF_CORES, PERF_WARMUP, PERF_REPEAT))

### 5.2 执行 CPU 版本

使用交易编号 `199016356` 运行 CPU 版本。程序从顺序表中筛选同一股票当日从首条记录至目标成交（含）的记录，依次累加成交量与成交额，计算累计 VWAP 并进行交易评价。

计时结果为正式运行的平均值。每次重新执行本单元都会清除上一轮 CPU、NPU 的对比记录，随后需继续运行第 5.3 节。


In [ ]:
# 每次重新运行 CPU 时，开始一轮新的对比，随后需要重新运行 NPU。
if 'PERF_RESULTS' not in globals():
    raise RuntimeError('请先运行第 5.1 节，设置本次运行参数。')
PERF_RESULTS.clear()
perf_cpu_bin = PERF_PROJECT / 'build_cpu/stock_vwap_cpu'
if not perf_cpu_bin.is_file():
    raise FileNotFoundError('请先完成第 4.2.2 节的 CPU 工程构建。')
perf_cpu_command = [str(perf_cpu_bin)] + _perf_arguments() + [
    '--cores', '1', '--csv', str(PERF_CPU_CSV)]
_perf_execute('CPU', perf_cpu_command, PERF_CPU_CSV)

### 5.3 执行 NPU 版本

使用相同的交易编号 `199016356`、数据文件、预热次数和正式计时次数运行 NPU 版本。第一阶段由多个计算核分别对各自数据块的成交量与成交额求和；第二阶段由一个计算核汇总局部结果，得到本次累计范围内的两项总和，再计算 VWAP 并进行交易评价。

下面通过项目运行脚本准备环境、构建并运行程序。图表读取 C++ 程序内部记录的计算耗时，脚本构建耗时不计入对比。如果当前环境未检测到 CANN 与 NPU 工具，则显示“未运行”，不生成 NPU 的性能数值。


In [ ]:
# 使用同一编号、同一输入与相同计时次数，实际运行 NPU 程序。
if 'PERF_RESULTS' not in globals():
    raise RuntimeError('请先运行第 5.1 节。')
PERF_RESULTS.pop('NPU', None)
perf_cpu_state = PERF_RESULTS.get('CPU', {}).get('state')
if perf_cpu_state == 'NOT_FOUND':
    PERF_RESULTS['NPU'] = {'state': 'NOT_FOUND', 'message': '交易编号不存在'}
    print('数据集中未找到默认交易编号 199016356，请核对本实验使用的 trades.txt。')
elif perf_cpu_state != 'OK':
    raise RuntimeError('请先完成第 5.2 节的 CPU 运行。')
elif not globals().get('HAS_ASCEND', False):
    PERF_RESULTS['NPU'] = {'state': 'SKIPPED', 'message': '当前环境未检测到 CANN 与 NPU 工具'}
    print('NPU 未运行：当前环境未检测到 CANN 与 NPU 工具。')
else:
    perf_script = PERF_PROJECT / 'scripts/run.sh'
    if not perf_script.is_file():
        raise FileNotFoundError('请先生成第 4 部分的 scripts/run.sh。')
    perf_env = os.environ.copy()
    perf_soc = str(globals().get('SOC_VERSION', '')).strip()
    if perf_soc:
        perf_env['SOC_VERSION'] = perf_soc
    else:
        perf_env.pop('SOC_VERSION', None)
    perf_npu_command = ['bash', str(perf_script)] + _perf_arguments() + [
        '--cores', str(PERF_CORES), '--device', str(int(globals().get('DEVICE_ID', 0))),
        '--csv', str(PERF_NPU_CSV)]
    _perf_execute('NPU', perf_npu_command, PERF_NPU_CSV, env=perf_env)

### 5.4 结果展示与性能分析

依次展示交易编号、交易记录条数、CPU 与 NPU 的执行结果表，以及平均耗时的标准横向条形图。交易记录条数指本次参与求和的累计范围内的记录数，即同一股票当日从首条记录至目标成交（含）的记录数。表中 VWAP 保留 8 位小数，平均耗时保留 3 位小数，单位为微秒（μs）；条形图从零开始，条形越短，表示本次计算耗时越少。

**计时范围：** 两条路径均对相同累计范围的成交量与成交额求和。CPU 计时包含两列数值的顺序累加；NPU 计时包含两阶段核函数启动、设备计算和同步等待。两者均不包含文件读取、交易查找、记录筛选、数值提取、数据上传、结果回传、校验和交易评价。

观察 CPU 与 NPU 的平均耗时，结合参与计算的记录数、核数以及启动和同步开销，分析本次性能差异。并行计算的收益随数据规模和运行条件变化，NPU 的耗时可能高于或低于 CPU。


In [ ]:
from html import escape
import math
from IPython.display import HTML, display


def _show_trade_performance():
    if 'PERF_RESULTS' not in globals():
        print('请先运行第 5.1～5.3 节。')
        return

    results = {}
    for name in ['CPU', 'NPU']:
        item = PERF_RESULTS.get(name, {})
        if item.get('state') == 'OK':
            if item.get('config') != PERF_CONFIG:
                raise ValueError('运行参数已变化，请从第 5.1 节重新运行。')

            row = item['row']
            if not math.isfinite(row['avg_us']) or row['avg_us'] < 0:
                raise ValueError('平均耗时应为非负的有限数值。')

            results[name] = row

    if not results:
        print('暂无有效结果，请先运行 CPU 和 NPU 版本。')
        return

    # 确保两条路径对应相同输入和累计结果。
    if len(results) == 2:
        fields = [
            'trade_id', 'n', 'volume_sum', 'turnover_sum', 'vwap', 'decision'
        ]
        if any(results['CPU'][key] != results['NPU'][key]
               for key in fields):
            raise ValueError('CPU 与 NPU 结果不一致，请先检查程序。')

    first = next(iter(results.values()))
    colors = {'CPU': '#4C72B0', 'NPU': '#55A868'}

    # 构建三线表。
    rows = []
    for name in ['CPU', 'NPU']:
        label = '<b><span style="color:%s">●</span> %s</b>' % (
            colors[name], name
        )
        row = results.get(name)

        if row is None:
            rows.append(
                '<tr><td>%s</td><td>—</td><td>未运行</td>'
                '<td>—</td><td>—</td></tr>' % label
            )
        else:
            cores = str(row['cores'])
            if name == 'NPU':
                cores += '（最大）'

            rows.append(
                '<tr><td>%s</td><td>%s</td><td>%s</td>'
                '<td>%.8f</td><td><b>%.3f</b></td></tr>' %
                (label, cores, escape(row['decision']),
                 row['vwap'], row['avg_us'])
            )

    # 确定从零开始的数值轴，自动选择刻度间隔。
    maximum = max(row['avg_us'] for row in results.values())
    target = maximum * 1.2 if maximum > 0 else 1.0
    raw_step = target / 5
    magnitude = 10 ** math.floor(math.log10(raw_step))

    step = next(
        value * magnitude
        for value in [1, 2, 2.5, 5, 10]
        if value * magnitude >= raw_step
    )
    tick_count = math.ceil(target / step)
    axis_max = tick_count * step

    # SVG 标准横向条形图。
    left, right, bottom = 80, 610, 170
    width = right - left

    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'viewBox="0 0 750 235" role="img" '
        'aria-label="CPU 与 NPU 平均耗时条形图">'
    ]

    # 数值轴刻度及浅色参考线。
    for i in range(tick_count + 1):
        value = i * step
        x = left + width * value / axis_max

        if i > 0:
            parts.append(
                '<line x1="%.2f" y1="22" x2="%.2f" y2="%d" '
                'stroke="#dedede" stroke-width="0.6"/>' %
                (x, x, bottom)
            )

        parts.append(
            '<line x1="%.2f" y1="%d" x2="%.2f" y2="%d" '
            'stroke="#333" stroke-width="0.8"/>' %
            (x, bottom, x, bottom + 5)
        )
        parts.append(
            '<text x="%.2f" y="192" text-anchor="middle">%g</text>' %
            (x, value)
        )

    # 条形长度表示实际平均耗时，末端标注数值。
    for i, name in enumerate(['CPU', 'NPU']):
        y = 60 + i * 65
        parts.append(
            '<text x="66" y="%d" text-anchor="end" '
            'dominant-baseline="middle">%s</text>' % (y, name)
        )

        row = results.get(name)
        if row is None:
            parts.append(
                '<text x="92" y="%d" '
                'dominant-baseline="middle">未运行</text>' % y
            )
            continue

        bar_width = width * row['avg_us'] / axis_max
        parts.append(
            '<rect x="%d" y="%d" width="%.3f" height="30" '
            'fill="%s" stroke="#333" stroke-width="0.6"/>' %
            (left, y - 15, bar_width, colors[name])
        )
        parts.append(
            '<text x="%.2f" y="%d" '
            'dominant-baseline="middle">%.3f</text>' %
            (left + bar_width + 8, y, row['avg_us'])
        )

    # 坐标轴与单位。
    parts.append(
        '<path d="M %d 22 V %d H %d" fill="none" '
        'stroke="#333" stroke-width="0.8"/>' %
        (left, bottom, right)
    )
    parts.append(
        '<text x="345" y="225" text-anchor="middle">'
        '平均耗时（μs）</text></svg>'
    )

    style = '''
<style>
.trade-result {
    max-width:820px;
    background:white;
    color:#222;
    font-family:Arial,"Microsoft YaHei",sans-serif;
    font-size:14px;
}
.trade-result p {
    margin:0 0 14px;
}
.trade-result table {
    width:100%;
    border-collapse:collapse;
    border-top:1.5px solid #222;
    border-bottom:1.5px solid #222;
}
.trade-result th,
.trade-result td {
    padding:10px 12px;
    text-align:right;
    border:0;
    background:white;
    white-space:nowrap;
    font-variant-numeric:tabular-nums;
}
.trade-result th:first-child,
.trade-result td:first-child {
    text-align:left;
}
.trade-result thead th {
    border-bottom:1px solid #222;
    font-weight:600;
}
.trade-result svg {
    display:block;
    width:100%;
    height:auto;
    margin-top:22px;
}
.trade-result svg text {
    font-size:13px;
    fill:#222;
}
</style>
'''

    content = '''
<div class="trade-result">
<p>交易编号：%s　　交易记录条数：%s</p>
<div style="overflow-x:auto">
<table>
<thead>
<tr>
    <th>执行版本</th>
    <th>核数 / 线程数</th>
    <th>交易评价</th>
    <th>VWAP（美元/股）</th>
    <th>平均耗时（μs）</th>
</tr>
</thead>
<tbody>%s</tbody>
</table>
</div>
%s
</div>
''' % (
        first['trade_id'],
        format(first['n'], ','),
        ''.join(rows),
        ''.join(parts)
    )

    display(HTML(style + content))


_show_trade_performance()

---
## 6. 实验总结

本实验基于同一种数据结构完成了股票交易评估：

- 使用顺序表保存完整成交记录，通过循环按序存入记录，并使用下标访问和顺序查找组织数据；
- 以目标记录的位置确定同一股票当日的累计范围，保持原有成交顺序；
- 从顺序表提取连续的成交量与成交额数组，作为 NPU 的数值输入；
- 将记录分给多个计算核，各核独立计算局部合计，再由第二个 NPU 核函数汇总；
- 使用 NPU 返回的总和计算 VWAP，从卖方视角完成交易评估；
- 核对两项整数总和和各核分工，并观察计算规模、核数及启动开销对执行时间的影响。

完成实验后，应能够解释“成交记录怎样组织”“各核分别处理哪些记录”“局部结果怎样合成总和”这三个问题。
